# 기후·토지이용 변화와 화분매개곤충: 서식지 이동, 꽃과의 공존 관계, 군집 구성 변화 예측

**(실습 `Workshop` `Notebook` — 단일 사례 계산 후 전체 분류군·2050년 `scenario` 비교 | 강의안 `SDM_HandsOn_Slides.pdf` `Slides 1–41` 대응)**

이 `Notebook`은 **제3~5차 전국자연환경조사(NES)**의 곤충·식물 출현 기록(부재는 기록되지 않은 `Presence-Only, PO` 자료)과 남한 내륙 1 km 격자(`84,576`개, `EPSG:5186`)의 기후·지형·토양·경관 환경자료 22종으로 아래 네 가지 생태학적 질문을 차례로 다룹니다(`Slides 1–2`). 서식 적합도는 출현 지점의 환경을 전국 무작위 배경 지점의 환경과 비교하는 방식(`Presence–Background, PB`; MaxEnt 계열)으로 추정합니다.

### 실습 진행 방식
각 단계(`Step 1` ~ `Step 4`)는 강의안(`SDM_HandsOn_Slides.pdf`) 순서에 맞춰 두 부분으로 구성됩니다.
1. **단일 사례 (`[실습]` cell)** — 기후 변화에 민감한 산지성 화분매개자 `FG03 뒤영벌속`을 중심 사례로 삼아(연평균기온 지도, 봄철 목본 밀원식물과의 관계, 두 조사 지점의 종조성 비교), 함수 내부 계산을 반복문 없이 `pandas`·`numpy`·`rasterio`·`matplotlib`으로 단계별로 계산합니다.
2. **전체 비교 (`[전체 비교]` cell)** — 같은 계산을 `sdm_core` 함수와 `for` 반복문으로 **5개 곤충 목**, **12개 기능군(`FG01`–`FG12`)**, **8개 식물–화분매개자 상호작용 쌍(및 274종 식물 × 139종 곤충 군집 합성)**, **4가지 군집 자료**에 적용해 분류군 간 반응 차이를 비교합니다.

---

1. **Step 1 — 분포 특성과 서식 환경 (`Slides 3–12`)**: 기능군별 출현 격자 수와 분포 고도, 실제 분포지에서 관찰된 여름 최고기온(`Bio5`) 상한·겨울 최저기온(`Bio6`) 하한(분포 기온 범위)을 확인하고, 2020년 기후·지형·경관 환경 구배를 지도화합니다.
2. **Step 2 — 서식 적합도 모형과 2050년 서식지 변화 (`Slides 13–24`)**: 환경요인에 대한 서식 적합도 반응(직선형·최적 범위형·문턱형; MaxEnt `L/Q/H`, Phillips & Dudík, 2008; Renner & Warton, 2013)을 추정하고, 학습에 쓰지 않은 50 km 권역에서도 분포를 맞히는지 검증합니다(공간 `block` 교차검증; Roberts et al., 2017; AUC, TSS [Allouche et al., 2006]). 이어 실제 분포지에서 관찰된 기온 한계를 넘는 곳을 서식 불가지로 처리하고(`Thermal_Limits.json`; Araújo et al., 2013), 2050년 서식지의 소실·유지·신규 확보 양상을 지도화합니다(Elsen & Tingley, 2015).
3. **Step 3 — 식물–화분매개자 공존 서식지 변화 (`Slides 25–29`)**: 화분매개자와 밀원·기주식물의 서식 적합지가 얼마나 겹치는지(Schoener's $D$; Warren et al., 2008) 비교하고, 2020년 공존 격자가 2050년에 `함께 유지`, `식물만 사라짐(먹이 부족)`, `화분매개자만 사라짐(수분 공백)`, `둘 다 사라짐` 중 어느 쪽으로 바뀌는지 분류하며, 전국 군집 수준 공존 피난처 보전 효과를 비교합니다(Schweiger et al., 2008; Schleuning et al., 2016; Briggs & Anderson, 2026; 개화·활동 시기 어긋남에 따른 식물 2차 절멸 위험은 Peng et al., 2025 참조).
4. **Step 4 — 군집 종조성 변화 (`Slides 30–39`)**: 두 지점(또는 두 시점) 사이의 종조성 차이(Sørensen 비유사도)를 '다른 종으로 바뀐 몫(종 교체)'과 '종이 빠져나가기만 한 몫(종 감소)'으로 나누고(Baselga, 2010; $\beta_{\text{sor}} = \beta_{\text{sim}} + \beta_{\text{sne}}$), 기온 등 환경 구배를 따라 종조성이 얼마나 빠르게 바뀌는지를 곡선(`I-spline`)으로 추정하는 일반화 비유사도 모형(GDM; Ferrier et al., 2007; Mokany et al., 2022; Cooke et al., 2026)을 종별 예측을 쌓아 만든 군집(SSDM)과 함께 적용합니다.

## 실행 환경 안내 (`MyBinder` 원격 실행 vs. `local` 실행)

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alanbseo/SDMworkshop/main?urlpath=lab/tree/SDM_HandsOn_Workshop_KO.ipynb)

이 실습은 별도 설치 없이 웹 `browser`에서 바로 여는 **`MyBinder` 원격 실행(권장 기본 방식)**과 본인 PC에 환경을 구성해 돌리는 **`local` 실행** 두 가지 방식을 모두 지원합니다.

| 구분 | 1. `MyBinder` 원격 실행 (설치 불필요 · 실습 기본) | 2. 개인 PC (`local`) 실행 |
| :--- | :--- | :--- |
| **실행 방법** | 아래 접속 주소(또는 상단 `Launch Binder` 배지)를 웹 `browser`에서 열면 이 `Notebook` 화면으로 바로 진입 | 저장소를 복제(`git clone`)하거나 배포본 압축을 푼 뒤 가상환경에서 `JupyterLab` 실행 |
| **사전 설치** | 불필요 (`Python 3.11`, `rasterio` 등 필수 `package`, `NanumGothic` 한글 글꼴, `data/` 사전 구성 완료) | `Python 3.10+` 및 `pip install -r requirements.txt` 직접 설치 필요 |
| **초기 대기** | 사전 생성본 기동 시 약 `20~40초` (저장소 갱신 직후 최초 생성 시 약 `3~5분`) | 설치 완료 후 즉시 실행 |
| **결과 보관** | **일시적 `session`**: 약 `10분`간 입력·실행이 없으면 자동 종료되며 변경 사항이 사라짐 → 종료 전 좌측 `file` 목록에서 `.ipynb` 우클릭 후 **`Download`(내려받기)** 필수 | 본인 저장장치의 `file`에 그대로 유지됨 |

- **`MyBinder` 직행 접속 주소**: https://mybinder.org/v2/gh/alanbseo/SDMworkshop/main?urlpath=lab/tree/SDM_HandsOn_Workshop_KO.ipynb
- **개인 PC (`local`) 실행 시 `terminal` 명령어**:
  ```bash
  git clone https://github.com/alanbseo/SDMworkshop.git
  cd SDMworkshop
  python3 -m venv .venv
  source .venv/bin/activate        # Windows: .venv\Scripts\activate
  pip install -r requirements.txt
  jupyter lab SDM_HandsOn_Workshop_KO.ipynb
  ```
  *(`macOS`·`Windows`는 운영체제 기본 한글 글꼴을 자동 인식하며, 개인 `Linux` 환경에서는 `fonts-nanum` 설치가 필요합니다.)*

## 0. 환경 설정
현재 실행 위치(`MyBinder` 원격 환경 또는 `local` PC)와 자료 경로(`WORKSHOP_ROOT`, `DATA_DIR`)를 확인하고, `scripts/sdm_core.py`에서 단일 사례용 기본 함수와 전체 비교용 묶음 함수를 불러오며, 운영체제에 맞는 한글 글꼴을 설정합니다.

In [ ]:
import os  # 운영체제 file·directory 경로 처리를 위한 표준 module 불러오기
import sys  # Python module 탐색 경로(sys.path) 제어를 위한 표준 module 불러오기
import json  # JSON 형식 metadata(Thermal_Limits.json) 입출력을 위한 module 불러오기
import numpy as np  # 수치 배열 및 행렬 연산을 위한 NumPy 불러오기
import pandas as pd  # 표 형식 DataFrame 처리를 위한 pandas 불러오기
import rasterio  # 공간 좌표계가 포함된 GeoTIFF raster 읽기를 위한 rasterio 불러오기
import matplotlib.pyplot as plt  # 공간 지도 및 통계 plot 시각화를 위한 matplotlib 불러오기
from matplotlib.colors import ListedColormap, BoundaryNorm  # 범주형 상태 지도를 위한 이산형 색상표·경계 정규화 class 불러오기

WORKSHOP_ROOT = os.path.abspath(".") if os.path.isdir("scripts") else os.path.abspath("..")  # 현재 위치에 scripts directory가 있으면 현재 경로를, 없으면 상위 경로를 root로 지정
sys.path.insert(0, os.path.join(WORKSHOP_ROOT, "scripts"))  # scripts directory를 Python module 탐색 경로 최상단에 추가

from sdm_core import (  # sdm_core engine에서 실습과 전체 비교에 필요한 상수와 함수 불러오기
    DATA_DIR, DRIVER_LABELS, DEFAULT_DRIVERS, GFO_COLORS,  # 자료 directory 경로, 22개 환경변수 label·목록, 표준 색상 palette
    load_pa_tables, load_env_scenario,  # 출현 격자 표 불러오기 함수 및 scenario별 22개 환경 raster 불러오기 함수
    build_maxent_features, fit_maxent_logistic, predict_suitability, evaluate_roc_tss,  # MaxEnt 기저 확장, 정규화 logistic 적합, 적합도 예측, ROC/TSS 평가 함수
    train_and_validate_sdm, compute_response_curve,  # 3-겹 공간 block 교차검증 wrapper 함수 및 반응곡선 산출 함수
    project_to_scenario, analyze_bipartite_overlap,  # 2050 scenario 서식지 예측 함수 및 식물-화분매개자 공존 분석 함수
    eval_ispline_3, fit_gdm_ispline, compute_baselga_temporal_beta,  # 3-매듭 I-spline 기저 계산, GDM 적합, Baselga beta 분할 함수
    vec_to_2d, plot_4state_map,  # 1차원 내륙 vector의 2차원 격자 복원 함수 및 서식지 변화 4유형 지도 시각화 함수
)  # sdm_core module import 구문 종료

DRIVER_LABELS_KO = {  # 22개 환경 예측변수 code를 국문 생태 의미·식별자·단위 label로 대응시키는 사전 정의
    "bioclim1": "연평균기온 (bioclim1, °C)",  # 연평균기온(전국 규모 기온 구배 지표)
    "bioclim2": "평균 일교차 (bioclim2, °C)",  # 월별 최고·최저기온 차의 연평균(대륙도·일교차 지표)
    "bioclim3": "등온성 (bioclim3, %)",  # 연교차 대비 일교차 비율(기온 계절 변동 완충 지표)
    "bioclim13": "최습월 강수량 (bioclim13, mm)",  # 여름철 집중호우·최습월 강수량
    "bioclim14": "최건월 강수량 (bioclim14, mm)",  # 겨울·봄 가뭄기 최건월 강수량
    "solarradiation": "일사량 (solarradiation, MJ/m²/d)",  # 지표면 도달 일평균 태양복사량(체온 조절·개화 energy)
    "relative_humidity": "상대습도 (relative_humidity, %)",  # 대기 상대습도(건조 stress 완충 지표)
    "VPD": "수증기압차 (VPD, hPa)",  # 포화 수증기압 대비 대기 수분 부족분(증발산·건조 stress)
    "AI": "건조지수 (AI)",  # 잠재증발산 대비 강수 비율 기반 습윤·건조 지수
    "WB_spr": "봄철 수분수지 (WB_spr, mm)",  # 봄철(3~5월) 강수-잠재증발산 수분수지(봄철 개화기 수분 가용성)
    "CWD": "기후적 수분부족량 (CWD, mm)",  # 생육기 누적 기후 수분 결핍량
    "elevation": "해발고도 (elevation, m)",  # 1 km 격자 평균 해발고도(m a.s.l.)
    "slope": "경사도 (slope, °)",  # 지형 경사각(배수·미소기후 이질성 지표)
    "soildeep3": "유효토심 (soildeep3)",  # 식물 뿌리 발달 및 지중 영소 벌류 서식 관련 유효토심
    "soildra1": "토양배수등급 (soildra1)",  # 토양 투수성 및 배수 양호도
    "soilstone": "토양자갈함량 (soilstone)",  # 표토·심토 자갈 함량(지중 영소 물리적 제약)
    "frac_0": "시가화 비율 (frac_0, 100m)",  # 100 m 토지피복에서 집계한 1 km 내 도시·건조환경 비율
    "frac_1": "농경지 비율 (frac_1, 100m)",  # 100 m 토지피복에서 집계한 1 km 내 농경지 비율
    "frac_2": "산림 비율 (frac_2, 100m)",  # 100 m 토지피복에서 집계한 1 km 내 산림 비율
    "frac_3": "초지 비율 (frac_3, 100m)",  # 100 m 토지피복에서 집계한 1 km 내 초지·개방지 비율
    "edge_FO": "산림-개방지 경계밀도 (edge_FO, km/km²)",  # 산림과 개방지(초지·농경지) 접점 길이 밀도(영소지-밀원 인접성)
    "shannon": "토지피복 다양성 (shannon, H')",  # 100 m 토지피복 유형들의 Shannon 다양성 지수(경관 이질성)
}  # 22개 환경 예측변수 국문 label 사전 정의 종료

import matplotlib.font_manager as fm  # 설치된 글꼴 탐색 및 직접 등록을 위한 font_manager 불러오기
nanum_ttf = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"  # MyBinder 등 Linux 환경의 나눔고딕 글꼴 기본 경로 지정
if os.path.exists(nanum_ttf):  # 해당 경로에 나눔고딕 글꼴 file이 존재하면(MyBinder·Linux 환경)
    fm.fontManager.addfont(nanum_ttf)  # 글꼴 목록 누락과 무관하게 matplotlib에 나눔고딕 즉시 등록
plt.rcParams["font.sans-serif"] = ["Apple SD Gothic Neo", "AppleGothic", "Malgun Gothic", "NanumGothic", "DejaVu Sans"]  # macOS·Windows·MyBinder(Linux) 환경 호환 한글 글꼴 우선순위 지정
plt.rcParams["axes.unicode_minus"] = False  # 한글 글꼴 사용 시 음수 부호(-)가 깨지지 않도록 설정
is_binder = ("BINDER_SERVICE_HOST" in os.environ) or (os.environ.get("USER") == "jovyan")  # 환경변수로 현재 실행 위치가 MyBinder 원격 환경인지 판별
run_env_label = "MyBinder (원격 실행)" if is_binder else "local (개인 PC / 자체 환경)"  # 판별 결과에 따른 실행 환경 표시 문자열 지정
avail_fonts = {f.name for f in fm.fontManager.ttflist}  # 현재 환경에서 matplotlib이 인식한 전체 글꼴 이름 집합 추출
active_font = next((fn for fn in plt.rcParams["font.sans-serif"] if fn in avail_fonts), "DejaVu Sans")  # 우선순위 목록 중 실제 선택된 한글 글꼴 확인
print(f"실행 환경: {run_env_label} | Python {sys.version.split()[0]} | 작업 경로: {WORKSHOP_ROOT} | 적용 글꼴: {active_font}")  # 실행 환경·Python version·작업 경로·적용 글꼴 출력
# Jupyter Notebook cell 바로 아래에 plot이 inline으로 출력되도록 magic 명령어 실행
%matplotlib inline

## Step 1-A. 출현 기록으로 본 분포 고도와 기온 범위 (`Slides 3–9`)
저온성 산지 화분매개자 **`FG03` 뒤영벌속(*Bombus*)**의 출현 격자 수, 출현지 평균 고도(전국 평균 대비), 분포지 여름 최고기온 상한·겨울 최저기온 하한을 출현 격자 표(`CSV`)와 기온 한계 표(`JSON`)에서 확인합니다.

In [ ]:
# [실습 1-A] FG03 뒤영벌속의 출현 격자, 분포 고도, 기온 한계(여름 상한·겨울 하한) 확인
pa_subgroups = pd.read_csv(os.path.join(DATA_DIR, "PA_Pollinator_Subgroups_1km.csv"))  # 12개 화분매개자 기능군의 1 km 출현 기록(PO) 격자 CSV file 읽기
bg_grid = pd.read_csv(os.path.join(DATA_DIR, "Background_Grid_1km.csv"))  # 남한 내륙 84,576개 1 km 배경 격자의 좌표·고도 표 읽기
with open(os.path.join(DATA_DIR, "Thermal_Limits.json"), encoding="utf-8") as f:  # 분류군별 분포 기온 한계(Bio5 여름 상한·Bio6 겨울 하한) JSON file 열기
    thermal_limits = json.load(f)  # JSON 내용을 Python dict 객체로 parsing하여 저장

print("PA_Pollinator_Subgroups_1km.csv 열(column) 목록:", list(pa_subgroups.columns))  # 출현 격자 DataFrame의 열 이름 목록 출력
print(f"남한 내륙 1 km 총 격자 수: {len(bg_grid):,}개 | 고도 결측치 수: {int(bg_grid['elevation_m'].isna().sum())}개")  # 전체 내륙 격자 수와 고도 결측 격자(0개) 검증 출력

bombus_df = pa_subgroups[pa_subgroups["subgroup_code"] == "FG03"]  # 반복문 없이 boolean indexing으로 FG03(뒤영벌속) 출현 행만 추출
bombus_ids = bombus_df["grid_id"].to_numpy(dtype=np.int32)  # 뒤영벌속이 출현한 1 km 격자 고유 번호(0~84,575)를 정수형 배열로 추출
bombus_tl = thermal_limits["subgroups"]["뒤영벌속"]  # 기온 한계 dict에서 뒤영벌속의 metadata 및 임계 온도 항목 조회

mean_elev_bombus = bg_grid.loc[bombus_ids, "elevation_m"].mean()  # 뒤영벌속 출현 격자들의 평균 해발고도(m) 계산
mean_elev_korea = bg_grid["elevation_m"].mean()  # 남한 전체 내륙 배경 격자(84,576개)의 평균 해발고도(m) 계산

print(f"FG03 뒤영벌속 출현 1 km 격자 수(NES 3~5차): {len(bombus_df):,}개 (5차 조사 포함: {int(bombus_df['has_5th'].sum()):,}개)")  # 총 출현 격자 수 및 5차 조사 출현 격자 수 출력
print(f"평균 해발고도 비교 — FG03 뒤영벌속: {mean_elev_bombus:.1f} m vs. 남한 내륙 전체 평균: {mean_elev_korea:.1f} m")  # 뒤영벌속 출현지와 전국 배경의 평균 고도 비교 출력
print(f"분포지에서 관찰된 기온 한계 — 여름 최고기온(Bio5) 상한 T_max: {bombus_tl['hot_limit_C']} °C | 겨울 최저기온(Bio6) 하한 T_min: {bombus_tl['cold_limit_C']} °C")  # 뒤영벌속의 여름 상한·겨울 하한 온도 출력
bombus_df.head(5)  # 추출된 뒤영벌속 출현 DataFrame의 상위 5개 행 미리보기 출력

`FG03 뒤영벌속(*Bombus*)`은 내륙 1 km 격자 중 `927`개 격자(5차 조사 `380`개)에서 확인되며, 출현 격자 평균 해발고도는 `309.3 m`로 남한 내륙 평균(`285.9 m`)보다 높고 분포지에서 관찰된 여름 최고기온 상한(`Bio5` $T_{\max}$)은 `33.41 °C`로 낮습니다. 다음 `cell`에서는 **12개 기능군(`FG01`–`FG12`)**의 분포 고도와 고온 내성을 비교합니다.

In [ ]:
# [전체 비교 1-A] 12개 화분매개자 기능군(FG01~FG12)의 출현 격자 수, 분포 고도, 기온 한계 비교
pa_orders, pa_subgroups, pa_plants, bg_grid, thermal_limits = load_pa_tables()  # 목·기능군·식물 출현 표, 배경 격자, 기온 한계를 한 번에 불러오기

SUBGROUP_LIST = [  # 분석 대상 12개 기능군의 (기능군 code, 국문명, 영문명, 상위 목 영문명) metadata list 정의
    ("FG03", "뒤영벌속", "Bombus", "Hymenoptera"),  # FG03: 벌목 뒤영벌속(저온성 산지 화분매개자)
    ("FG04", "꿀벌속", "Apis", "Hymenoptera"),  # FG04: 벌목 꿀벌속(광온성 사회성 꿀벌류)
    ("FG05", "기타 야생꿀벌과", "Other Apidae", "Hymenoptera"),  # FG05: 벌목 기타 야생꿀벌과(어리호박벌·알락꽃벌류 등)
    ("FG06", "단독성 야생벌류", "Solitary Bees", "Hymenoptera"),  # FG06: 벌목 단독성 야생벌류(봄철 활동 단독성 벌류)
    ("FG01", "꽃등에과", "Syrphidae", "Diptera"),  # FG01: 파리목 꽃등에과(광범위 일반종 꽃등에류)
    ("FG02", "재니등에과", "Bombyliidae", "Diptera"),  # FG02: 파리목 재니등에과(개방 초지 선호 재니등에류)
    ("FG07", "온대 노린재류", "Temperate Hemiptera", "Hemiptera"),  # FG07: 노린재목 온대 노린재류(내륙 초본·관목 방문 노린재류)
    ("FG08", "큰별노린재과", "Southern Largidae", "Hemiptera"),  # FG08: 노린재목 큰별노린재과(남해안·도서 아열대성 노린재류)
    ("FG09", "호랑나비과", "Papilionidae", "Lepidoptera"),  # FG09: 나비목 호랑나비과(산림성 대형 나비류)
    ("FG10", "네발나비과", "Nymphalidae", "Lepidoptera"),  # FG10: 나비목 네발나비과(임연부·초지 네발나비류)
    ("FG11", "기타 나비·나방류", "Other Lepidoptera", "Lepidoptera"),  # FG11: 나비목 기타 나비·나방류(개방지 나비 및 주간 활동 나방류)
    ("FG12", "하늘소과", "Cerambycidae", "Coleoptera"),  # FG12: 딱정벌레목 하늘소과(산림 고사목 의존 꽃하늘소류)
]  # 12개 기능군 metadata list 정의 종료

rows_1a = []  # 기능군별 요약 통계 dict를 누적할 빈 list 초기화
for scode, skr, sen, oen in SUBGROUP_LIST:  # 12개 기능군 metadata를 순서대로 순회
    sub = pa_subgroups[pa_subgroups["subgroup_kr"] == skr]  # 현재 기능군 국문명(skr)과 일치하는 출현 격자 행만 추출
    tl = thermal_limits["subgroups"][skr]  # 현재 기능군의 분포 기온 상·하한 정보 조회
    rows_1a.append({  # 현재 기능군의 집계 결과를 dict로 구성하여 list에 추가
        "기능군_code": scode,  # 기능군 고유 식별자(FG01~FG12)
        "상위_목": oen,  # 상위 곤충 목 영문명
        "기능군_국문명": skr,  # 기능군 국문 명칭
        "기능군_영문명": sen,  # 기능군 영문 명칭
        "출현격자_3_5차": len(sub),  # 제3~5차 전국자연환경조사 통합 출현 1 km 격자 수
        "출현격자_5차": int(sub["has_5th"].sum()),  # 제5차 조사에서 확인된 1 km 격자 수
        "평균고도_m": round(float(bg_grid.loc[sub["grid_id"], "elevation_m"].mean()), 1),  # 출현 격자들의 평균 해발고도(소수 첫째 자리 반올림)
        "하계상한_Bio5_C": tl["hot_limit_C"],  # 최난월 최고기온(Bio5) 실현 상한 임계치(°C)
        "동계하한_Bio6_C": tl["cold_limit_C"],  # 최한월 최저기온(Bio6) 보수적·경험적 하한 임계치(°C)
    })  # 단일 기능군 요약 dict 추가 완료

pd.DataFrame(rows_1a)  # 누적된 12개 기능군 요약 list를 DataFrame으로 변환하여 출력

같은 벌목(`Hymenoptera`) 내에서도 `FG03 뒤영벌속`(`Bio5` 상한 `33.41 °C`, 출현 격자 평균 고도 `309.3 m`)과 `FG04 꿀벌속`(`41.53 °C`, `272.0 m`)의 고온 내성과 분포 고도가 뚜렷이 대비되며, 노린재목에서도 내륙 전역에 분포하는 `FG07 온대 노린재류`(`2,025`개 격자, 상한 `33.41 °C`, 동계 하한 `-23.33 °C`)와 남해안·남부 산록에 국한된 아열대성 `FG08 큰별노린재과`(`124`개 격자, 상한 `35.86 °C`, 동계 하한 `-13.96 °C`)가 상반된 기후 지위를 보입니다.

## Step 1-B. 기후·지형·경관 환경 구배 (`Slides 10–12`)
- **공통 핵심 예측변수(`SDM & GDM` 공통, 22개 `raster`)**: 생물기후(`bioclim1, 2, 3, 13, 14`), 대기·복사(`solarradiation, relative_humidity, VPD`), 수분수지(`AI, WB_spr, CWD`), 지형·토양(`elevation, slope, soildeep3, soildra1, soilstone`), 100 m 토지피복·경관구조(`frac_0..3, edge_FO, shannon`)로 구성됩니다. **SDM**에서는 격자별 환경값 $x_i$의 `L/Q/H` 기저변환으로 입력되고, **GDM**에서는 지점 쌍 간 3-매듭 `I-spline` 변환 차이 $|\Delta I_k(x)|$로 입력됩니다.
- **모형별 보조 입력자료**: `bioclim5`, `bioclim6`(여름 최고·겨울 최저기온; 분포 기온 한계를 넘는 격자를 서식 불가지로 처리하는 데 사용) 및 `frac_6`(수역 비율 $\ge 0.5$ 격자는 육상 서식지에서 제외)는 모형 적합 후 **SDM에만** 덧씌우는 제약 조건이며, 지점 쌍 간 `Euclidean` 지리적 거리 `Geo_Distance`(`x_5186, y_5186`)는 **GDM 전용** 공간 `spline` 예측변수입니다(**SDM**에서는 50 km 공간 `block` 교차검증 분할에만 사용).

연평균기온(`bioclim1.tif`)에 내륙 `mask`(`land_mask_1km.tif`, `84,576`개 격자)를 적용해 결측치를 확인하고, `FG03 뒤영벌속` 출현 지점을 중첩해 출현 기온대를 확인합니다.

In [ ]:
# [실습 1-B] 연평균기온 지도(bioclim1.tif) 위에 FG03 뒤영벌속 출현 지점을 겹쳐 서식 기온대 확인
with rasterio.open(os.path.join(DATA_DIR, "land_mask_1km.tif")) as src_mask:  # 남한 1 km 육지 mask GeoTIFF file 열기
    mask_raw = src_mask.read(1)  # 첫 번째 band를 2차원 배열(602행 x 342열)로 읽기
    mask_2d = (~np.isnan(mask_raw)) & (mask_raw > 0)  # NaN이 아니고 양수(>0)인 내륙 유효 화소만 True로 표시하는 boolean mask 생성

with rasterio.open(os.path.join(DATA_DIR, "env_2020", "bioclim1.tif")) as src_b1:  # 2020년 기준 연평균기온(bioclim1) GeoTIFF file 열기
    bio1_2d_raw = src_b1.read(1).astype(np.float32)  # 첫 번째 band를 32-bit 실수형 2차원 배열로 읽기

bio1_1d = bio1_2d_raw[mask_2d]  # 2차원 배열에 육지 mask를 적용하여 길이 84,576의 1차원 내륙 vector 추출
print(f"2D raster 크기: {bio1_2d_raw.shape} | 내륙 유효 1 km 격자 수: {mask_2d.sum():,}개 | 내륙 결측치(NaN) 수: {int(np.isnan(bio1_1d).sum())}개")  # 격자 차원, 유효 화소 수, 결측치 0개 검증 출력
print(f"연평균기온 비교 — FG03 뒤영벌속 출현 격자: {bio1_1d[bombus_ids].mean():.2f} °C vs. 내륙 전체 평균: {bio1_1d.mean():.2f} °C")  # 뒤영벌속 출현지와 전국 내륙의 연평균기온 비교 출력

bio1_map = np.full(mask_2d.shape, np.nan, dtype=np.float32)  # 바다·영역 외 화소를 NaN으로 채운 빈 2차원 지도 배열 초기화
bio1_map[mask_2d] = bio1_1d  # 내륙 유효 mask 위치에만 1차원 연평균기온 값을 대입하여 2차원 지도 복원

fig, ax = plt.subplots(figsize=(4.8, 5.2))  # 가로 4.8 inch, 세로 5.2 inch 크기의 단일 지도 figure와 축 생성
im = ax.imshow(bio1_map, cmap="RdYlBu_r", vmin=np.nanpercentile(bio1_map, 2), vmax=np.nanpercentile(bio1_map, 98))  # 상·하위 2% 분위수로 색 대비를 맞춘 연평균기온 raster 표시
ax.scatter(bombus_df["col"], bombus_df["row"], s=3.5, c="black", alpha=0.5, label="FG03 뒤영벌속 출현(PO)")  # raster 열(col)·행(row) 좌표 위에 뒤영벌속 출현 지점 산점도 중첩
ax.set_title("2020 연평균기온(bioclim1) 및 FG03 출현 격자", fontsize=10.5, fontweight="bold")  # 지도 상단 제목 설정
ax.axis("off")  # 격자 행·열 눈금 테두리 숨김 처리
ax.legend(loc="lower right", fontsize=8)  # 우측 하단에 출현 지점 범례 표시
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03, label="°C")  # 우측에 온도 단위(°C) colorbar 추가
plt.tight_layout()  # 제목·colorbar 여백이 겹치지 않도록 자동 정렬
plt.show()  # 완성된 지도를 화면에 출력

모든 1 km GeoTIFF `raster`는 `(602, 342)` 2차원 격자 중 정확히 `84,576`개의 내륙 유효 화소(결측치 `0`개)로 대응되며, `FG03 뒤영벌속` 출현 격자의 연평균기온(`12.00 °C`)은 내륙 전체 평균(`12.28 °C`)보다 서늘한 백두대간·산지 축을 따라 분포합니다. 다음 `cell`에서는 `load_env_scenario("env_2020")`으로 **22개 환경 예측변수 전체**를 불러와 기온·봄철 수분수지·고도·산림 비율·산림–개방지 경계밀도 5개 구배를 비교합니다.

In [ ]:
# [전체 비교 1-B] 2020년 22개 환경 예측변수를 불러오고 핵심 5개 환경 구배를 지도로 비교
env_2020 = load_env_scenario("env_2020")  # data/env_2020 directory에서 22개 예측변수 행렬(X_raw)과 Bio5·Bio6·수역 비율을 한꺼번에 불러오기
print("불러온 X_raw 행렬 크기 (내륙 격자 수, 예측변수 수):", env_2020["X_raw"].shape)  # (84576, 22) 형태의 예측변수 행렬 차원 출력

show_vars = [  # 시각화로 비교할 핵심 5개 환경변수의 (변수 code, colormap, 국문 제목) list 정의
    ("bioclim1", "RdYlBu_r", "연평균기온 (bioclim1, °C)"),  # 거시 기온 구배를 대표하는 연평균기온
    ("WB_spr", "BrBG", "봄철 수분수지 (WB_spr, mm)"),  # 봄철 개화기 수분 가용성을 나타내는 봄철 수분수지
    ("elevation", "terrain", "해발고도 (elevation, m)"),  # 산악 지형 구배를 나타내는 해발고도
    ("frac_2", "Greens", "산림 비율 (frac_2, 100m)"),  # 100 m 해상도에서 집계한 1 km 격자 내 산림 피복 비율
    ("edge_FO", "PuBuGn", "산림-개방지 경계밀도 (edge_FO, km/km²)"),  # 산림과 초지·농경지 접점 밀도(영소지-먹이원 인접성 지표)
]  # 핵심 5개 환경변수 시각화 목록 정의 종료

fig, axes = plt.subplots(1, 5, figsize=(15.5, 3.6))  # 1행 5열 다중 지도 panel 생성
for ax, (vcode, cm, ktitle) in zip(axes, show_vars):  # 각 subplot 축과 환경변수 설정을 순서대로 짝지어 순회
    arr = vec_to_2d(env_2020["layers"][vcode], mask_2d)  # 1차원 내륙 환경변수 vector를 2차원 지도 배열(602x342)로 복원
    im = ax.imshow(arr, cmap=cm, vmin=np.nanpercentile(arr, 2), vmax=np.nanpercentile(arr, 98))  # 2~98 백분위수 범위로 환경 구배 지도 표시
    ax.set_title(ktitle, fontsize=9.2, fontweight="bold")  # 각 panel 상단에 국문 환경변수 제목 표시
    ax.axis("off")  # 지도 축 눈금 및 테두리 비활성화
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)  # 각 panel 우측에 변수 scale colorbar 부착
plt.tight_layout()  # 5개 panel 간 간격 자동 조정
plt.show()  # 다중 환경 구배 지도 출력

22개 기본 예측변수 행렬 `(84576, 22)`이 정상적으로 불러와졌으며, 거시 기후(`bioclim1`, `WB_spr`)·지형(`elevation`) 구배와 더불어 100 m 토지피복에서 산출한 산림 비율(`frac_2`) 및 산림–개방지 경계밀도(`edge_FO`, `km/km²`)가 국지적 서식지 이질성을 세밀하게 포착합니다.

## Step 2-A. 서식 적합도 모형: 환경 반응 추정과 미조사 지역 예측력 검증 (`Slides 13–16`)
`train_and_validate_sdm` 함수의 내부 절차를 **`FG03 뒤영벌속`**으로 단계별 구현합니다. 출현 격자와 전국 배경 격자의 환경 차이로 모형을 적합하고, 학습에서 제외한 권역에서 예측력을 검증합니다.
1. 내륙 전체에서 `10,000`개의 배경 격자(`y = 0`)를 무작위 추출해 출현 격자(`y = 1`)와 비교합니다. 배경 격자는 부재지가 아니라 전국 가용 환경의 표본입니다.
2. 남한 전역을 **`50 km × 50 km` 권역(공간 `block`)**(`EPSG:5186` `m` 좌표 $\div 50{,}000$)으로 나누고, 1/3 권역(**`Fold 0`**)을 학습에서 제외해 검증에만 사용합니다. 인접 격자 간 공간 자기상관 때문에 무작위 분할 검증은 예측력을 과대평가합니다(Roberts et al., 2017).
3. 각 환경변수를 세 가지 반응 형태로 변환합니다: 값에 비례해 증감하는 **직선형(L)**, 중간값에서 최대가 되는 **최적 범위형(Q)**, 특정 값 이상(이하)에서 변하는 **문턱형(`hinge`, H)** (`22 선형 + 22 이차 + 44 hinge = 88개 특징`). 이때 표준화 기준(평균·표준편차·문턱 위치, `scaler`)은 **학습 권역(`Fold 1 & 2`)에서만** 계산해 검증 권역 정보의 유입을 차단합니다(data leakage 방지).
4. 출현·배경 집단에 같은 총 가중치를 주고, 과적합을 억제하는 L1/L2 정규화 `logistic` 회귀(`fit_maxent_logistic`)로 적합합니다. 검증 권역에서는 출현지를 배경보다 높게 평가하는 능력(**AUC**)과 서식지/비서식지를 가르는 최적 기준값(**Max-TSS 임계치 $\tau$**; Allouche et al., 2006)을 구합니다.

In [ ]:
# [실습 2-A1] FG03 뒤영벌속 서식 적합도 모형을 학습 권역에서 만들고, 학습에 쓰지 않은 50 km 권역에서 예측력 검증
rng = np.random.default_rng(42)  # 재현 가능한 난수 생성을 위해 seed(42) 고정 난수 생성기 초기화
pres_ids = np.unique(bombus_ids)  # 뒤영벌속 출현 격자 ID 중복 제거 및 오름차순 정렬
bg_ids = rng.choice(bg_grid["grid_id"].to_numpy(dtype=np.int32), size=10000, replace=False)  # 내륙 84,576개 격자 중 10,000개 배경 격자를 비복원 무작위 추출

# 1) EPSG:5186 평면직각좌표(m)로부터 50 km x 50 km 공간 block 번호 할당
xs = bg_grid["x_5186"].to_numpy()  # 전 격자의 동서 방향 평면직각좌표(m) 배열 추출
ys = bg_grid["y_5186"].to_numpy()  # 전 격자의 남북 방향 평면직각좌표(m) 배열 추출
block_code = (np.floor((xs - xs.min()) / 50000.0).astype(int) * 100 + np.floor((ys - ys.min()) / 50000.0).astype(int))  # 50,000 m 단위로 나눈 행·열 몫을 결합해 고유 공간 block 번호 생성

uniq_blocks = np.unique(block_code)  # 생성된 고유 50 km 공간 block 번호 목록 추출
rng.shuffle(uniq_blocks)  # 공간 block 순서를 무작위로 섞기
block_to_fold = {b: (i % 3) for i, b in enumerate(uniq_blocks)}  # 각 공간 block을 0, 1, 2번 교차검증 fold(3-fold)에 균등 배정
cell_fold = np.array([block_to_fold[b] for b in block_code], dtype=int)  # 개별 1 km 격자마다 소속 공간 block의 fold 번호(0, 1, 2) 대응

# 2) 기저함수 scaler 적합 전에 먼저 Fold 0(검증 집합)과 Fold 1·2(훈련 집합)를 엄격히 분리
X_raw_2020 = env_2020["X_raw"]  # 2020년 기준 22개 원시 환경변수 행렬(84,576 x 22) 참조
train_mask = cell_fold != 0  # Fold 1과 Fold 2에 속하는 훈련 영역 격자만 True로 표시
_, scaler_train = build_maxent_features(X_raw_2020[train_mask], scaler=None, use_hinge=True)  # 오직 훈련 영역 격자만으로 L/Q/H 평균·표준편차·hinge 매듭 scaler 산출

tr_pres = pres_ids[cell_fold[pres_ids] != 0]  # 훈련 block(Fold 1·2)에 속한 출현 격자 ID 추출
te_pres = pres_ids[cell_fold[pres_ids] == 0]  # 독립 검증 block(Fold 0)에 속한 출현 격자 ID 추출
tr_bg = bg_ids[cell_fold[bg_ids] != 0]  # 훈련 block(Fold 1·2)에 속한 배경 격자 ID 추출
te_bg = bg_ids[cell_fold[bg_ids] == 0]  # 독립 검증 block(Fold 0)에 속한 배경 격자 ID 추출

X_tr_pres, _ = build_maxent_features(X_raw_2020[tr_pres], scaler=scaler_train)  # 훈련 출현 격자 환경변수를 훈련 scaler 기준으로 88개 L/Q/H 특징으로 변환
X_tr_bg, _ = build_maxent_features(X_raw_2020[tr_bg], scaler=scaler_train)  # 훈련 배경 격자 환경변수를 훈련 scaler 기준으로 88개 L/Q/H 특징으로 변환
X_te_pres, _ = build_maxent_features(X_raw_2020[te_pres], scaler=scaler_train)  # 검증 출현 격자 환경변수를 훈련 scaler 기준으로 변환(data leakage 차단)
X_te_bg, _ = build_maxent_features(X_raw_2020[te_bg], scaler=scaler_train)  # 검증 배경 격자 환경변수를 훈련 scaler 기준으로 변환(data leakage 차단)
print(f"기저함수 확장 완료: 원시 변수 {X_raw_2020.shape[1]}개 -> 표준화 L/Q/H 특징 {X_tr_pres.shape[1]}개")  # 22개 원시 변수가 88개 파생 특징으로 확장되었음을 출력
print(f"Fold-0 분할 현황: 훈련 출현={len(tr_pres)}개, 검증 출현={len(te_pres)}개 | 훈련 배경={len(tr_bg)}개, 검증 배경={len(te_bg)}개")  # 훈련·검증 표본 크기 출력

# 3) 훈련 block에서 정규화 logistic SDM 적합 후 독립 검증 block(Fold 0)에서 예측력 평가
params_fold0 = fit_maxent_logistic(X_tr_pres, X_tr_bg, beta_reg=1.5)  # 훈련 집합으로 출현·배경 동일 비중 가중 L1/L2 정규화(beta_reg=1.5) logistic 회귀 계수 추정
prob_te_pres = predict_suitability(params_fold0, X_te_pres, env_2020["water_frac"][te_pres])  # 검증 출현 격자의 상대 서식 적합도(0~1, 수역 비율 보정 포함) 예측
prob_te_bg = predict_suitability(params_fold0, X_te_bg, env_2020["water_frac"][te_bg])  # 검증 배경 격자의 상대 서식 적합도(0~1, 수역 비율 보정 포함) 예측

fold0_metrics = evaluate_roc_tss(prob_te_pres, prob_te_bg)  # 검증 집합 예측값으로부터 ROC-AUC, 최대 TSS, 최적 이진화 임계치(tau) 계산
print(f"[FG03 독립 검증 Fold 0] AUC = {fold0_metrics['auc']:.3f}, Max-TSS = {fold0_metrics['tss']:.3f}, 최적 임계치 tau = {fold0_metrics['threshold']:.3f}")  # 검증 성능 지표 출력

22개 원시 환경변수가 `88`개 표준화 `L/Q/H` 기저 특징으로 확장되었으며, 공간적으로 완전히 분리된 단일 50 km 검증 `block`(`Fold 0`, 훈련 출현 `653`개·검증 출현 `274`개)에서도 `FG03 뒤영벌속` 모형이 우연 수준(`AUC = 0.50`)을 뚜렷이 상회하는 공간 일반화 성능(`AUC = 0.618`, `Max-TSS = 0.191`, 최적 임계치 `tau = 0.451`; 3개 권역을 모두 교대 검증한 `Step 2-B` 요약표 및 `Slide 24`의 3-겹 통합 공간 CV `AUC`는 `0.628`)을 보입니다.

이어서 나머지 21개 변수를 전국 중앙값(median)으로 고정하고 `bioclim1`(연평균기온)만 하위 `1%`에서 상위 `99%`까지 100단계로 바꾸어 **기온 반응곡선(`marginal response curve`)**을 계산합니다.

In [ ]:
# [실습 2-A2] 다른 조건을 고정했을 때 연평균기온(bioclim1)에 따른 FG03 뒤영벌속 서식 적합도 변화(반응곡선) 계산
X_all_feat, scaler_full = build_maxent_features(X_raw_2020, scaler=None, use_hinge=True)  # 최종 반응곡선 산출을 위해 전국 내륙 격자 전체에 대해 L/Q/H 기저 특징 및 scaler 생성
params_bombus = fit_maxent_logistic(X_all_feat[pres_ids], X_all_feat[bg_ids], beta_reg=1.5)  # 전체 출현 격자와 10,000개 배경 격자로 뒤영벌속 최종 logistic 모형 적합

var_code = "bioclim1"  # 반응곡선을 살펴볼 대상 환경변수 code(연평균기온) 지정
var_idx = DEFAULT_DRIVERS.index(var_code)  # 22개 환경변수 목록에서 bioclim1이 위치한 열 index 조회

X_profile = np.tile(np.median(X_raw_2020, axis=0), (100, 1))  # 22개 변수의 전국 중앙값 vector를 100개 행으로 복제한 가상 환경 profile 행렬 생성
temp_axis = np.linspace(np.percentile(X_raw_2020[:, var_idx], 1), np.percentile(X_raw_2020[:, var_idx], 99), 100)  # 연평균기온의 1% 분위수부터 99% 분위수까지 등간격 100개 온도 구간 생성
X_profile[:, var_idx] = temp_axis  # 가상 profile 행렬에서 bioclim1 열만 100단계 온도 값으로 대체(나머지 21개 변수는 중앙값 유지)

X_profile_feat, _ = build_maxent_features(X_profile, scaler=scaler_full)  # 100행 가상 profile을 전체 scaler 기준으로 88개 L/Q/H 특징 행렬로 변환
suit_curve = predict_suitability(params_bombus, X_profile_feat)  # 온도 변화에 따른 뒤영벌속의 상대 서식 적합도(0~1) 곡선 예측

fig, ax = plt.subplots(figsize=(5.6, 3.3))  # 반응곡선 시각화를 위한 단일 plot figure 생성
ax.plot(temp_axis, suit_curve, lw=2.4, color=GFO_COLORS["slate"], label="FG03 뒤영벌속 상대 적합도")  # 연평균기온 축 대비 서식 적합도 곡선 그리기
ax.axhline(fold0_metrics["threshold"], color=GFO_COLORS["terracotta"], ls="--", label=f"임계치 tau = {fold0_metrics['threshold']:.2f}")  # Max-TSS 기준 서식 적합/부적합 기준선(점선) 표시
ax.set_xlabel("연평균기온 (bioclim1, °C)")  # 가로축 label 설정
ax.set_ylabel("상대 서식 적합도 (Suitability)")  # 세로축 label 설정
ax.set_title("연평균기온에 대한 서식 적합도 반응 (FG03 뒤영벌속)", fontweight="bold")  # plot 제목 설정
ax.set_ylim(0, 1)  # 세로축 적합도 범위를 0~1로 고정
ax.legend()  # 범례 표시
plt.tight_layout()  # plot 여백 자동 정렬
plt.show()  # 반응곡선 plot 출력

`FG03 뒤영벌속`의 연평균기온 반응곡선은 저온 산지 구간에서 임계치(`tau = 0.45`) 이상의 높은 적합도를 유지하다가 연평균기온이 높아지는 저지대 구간으로 갈수록 단조 감소합니다. 다음 `cell`의 `train_and_validate_sdm(...)` 함수는 위의 검증을 3개 권역(`fold`)이 번갈아 검증 권역이 되도록 반복하며, 모형 내 상대 변수 중요도(`%`, 22개 변수 합계 = `100%`, 균등 기준선 = `4.5%`)를 표준화 단위 표준편차(1 SD) 기준 `L/Q/H` 계수 절댓값 합($S_k = |\hat{\beta}_{k,\text{L}}| + |\hat{\beta}_{k,\text{Q}}| + |\hat{\beta}_{k,\text{H1}}| + |\hat{\beta}_{k,\text{H2}}|$)의 비중으로 산출합니다(이는 결정계수 $R^2$나 순열 AUC 감소량과는 다른 **`log-odds`** 민감도 비중입니다). `TARGET_ORDER_EN` 값을 바꾸면 5개 곤충 목 중 원하는 분류군의 교차검증 ROC 곡선과 상위 3개 반응곡선을 바로 확인할 수 있습니다.

In [ ]:
# [전체 비교 2-A] 선택한 곤충 목의 예측력(3개 권역 교대 검증)과 영향력이 가장 큰 환경변수 3개의 반응곡선
ORDER_EN_TO_KR = {  # 곤충 5개 목의 영문명을 dataset 국문명과 대응시키는 dict 정의
    "Hymenoptera": "벌목",  # 벌목(뒤영벌·꿀벌·단독성 야생벌류 포함)
    "Diptera": "파리목",  # 파리목(꽃등에과·재니등에과 포함)
    "Lepidoptera": "나비목",  # 나비목(호랑나비과·네발나비과·기타 나비·나방류 포함)
    "Hemiptera": "노린재목",  # 노린재목(온대 노린재류·큰별노린재과 포함)
    "Coleoptera": "딱정벌레목",  # 딱정벌레목(하늘소과 포함)
}  # 영문-국문 목 대응 dict 정의 종료
TARGET_ORDER_EN = "Hymenoptera"  # 분석할 대상 목 지정 (Diptera, Lepidoptera, Hemiptera, Coleoptera로 변경하여 실습 가능)
TARGET_ORDER_KR = ORDER_EN_TO_KR[TARGET_ORDER_EN]  # 선택한 영문 목에 대응하는 국문 목 명칭 조회

order_pres_ids = pa_orders.loc[pa_orders["order_kr"] == TARGET_ORDER_KR, "grid_id"].to_numpy()  # 선택한 목의 출현 1 km 격자 ID 배열 추출
order_model = train_and_validate_sdm(order_pres_ids, env_2020, bg_grid, beta_reg=1.5, use_hinge=True)  # 3-겹 50 km 공간 block 교차검증 및 전체 자료로 최종 모형 적합
cv = order_model["cv_metrics"]  # 3개 fold 통합 공간 교차검증 평가 지표(AUC, TSS, 임계치, ROC 좌표) 추출
print(f"[{TARGET_ORDER_EN} ({TARGET_ORDER_KR})] 3-겹 50 km 공간 block CV: AUC = {cv['auc']:.3f}, TSS = {cv['tss']:.3f}, 임계치 = {cv['threshold']:.3f}")  # 교차검증 성능 요약 출력

top_vars = sorted(order_model["var_imp"].items(), key=lambda x: -x[1])  # 22개 예측변수를 상대 중요도(%) 내림차순으로 정렬

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))  # 좌측 ROC 곡선 1개 + 상위 3개 변수 반응곡선용 1행 4열 panel 생성
axes[0].plot(cv["fpr_curve"], cv["tpr_curve"], lw=2, color=GFO_COLORS["ochre"], label=f"AUC={cv['auc']:.3f}")  # 첫 번째 panel에 공간 교차검증 ROC 곡선 그리기
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)  # 무작위 예측 기준선(대각선 점선, AUC=0.5) 표시
axes[0].set_title(f"{TARGET_ORDER_KR} ({TARGET_ORDER_EN}) 공간 CV ROC", fontsize=9.5, fontweight="bold")  # 첫 번째 panel 제목 설정
axes[0].set_xlabel("위양성률 (FPR)")  # ROC 가로축 label 설정
axes[0].set_ylabel("진양성률 (TPR)")  # ROC 세로축 label 설정
axes[0].legend()  # ROC panel 범례 표시

for i, (vcode, vimp) in enumerate(top_vars[:3]):  # 상대 중요도 상위 3개 예측변수에 대해 순서대로 순회
    gx, gy = compute_response_curve(order_model, env_2020, vcode)  # 해당 예측변수의 구간 값(gx)과 적합도 반응곡선(gy) 계산
    axes[i + 1].plot(gx, gy, lw=2.2, color=GFO_COLORS["slate"])  # (i+1)번째 panel에 반응곡선 그리기
    axes[i + 1].axhline(order_model["threshold"], color=GFO_COLORS["terracotta"], ls="--")  # Max-TSS 이진화 임계치 기준선(점선) 표시
    axes[i + 1].set_title(f"{DRIVER_LABELS_KO[vcode]} ({vimp:.1f}%)", fontsize=9.0, fontweight="bold")  # 국문 변수명과 상대 중요도(%)를 panel 제목으로 표시
    axes[i + 1].set_ylim(0, 1)  # 세로축 적합도 범위를 0~1로 통일
plt.tight_layout()  # 4개 subplot 간격 자동 정렬
plt.show()  # ROC 및 상위 3개 반응곡선 출력

pd.DataFrame([(k, DRIVER_LABELS_KO[k], round(v, 2)) for k, v in top_vars[:6]], columns=["변수_code", "환경_예측변수", "상대_중요도_pct"])  # 상위 6개 핵심 환경변수 국문 중요도 표 출력

`벌목(Hymenoptera)` 통합 모형은 50 km 공간 `block` 교차검증에서 `AUC = 0.621`, `TSS = 0.195`(`임계치 = 0.441`)를 기록하며, 100 m 토지피복 다양성(`shannon`, `16.4%`), 산림–개방지 경계밀도(`edge_FO`, `13.8%`), 일사량(`solarradiation`, `10.0%`), 산림 비율(`frac_2`, `9.1%`) 순으로 높은 상대 중요도를 차지합니다.

## Step 2-B. 여름철 고온 한계를 반영한 2050년 분포 변화와 산지 방향 후퇴 (`Slides 17–24`)
2020년 자료로 만든 모형에 2050년 환경(`Biodiversity-SSP126`: 생태보전형 토지이용 + 저탄소 기후, `BAU-SSP585`: 현 추세 토지이용 + 고탄소 기후)을 넣어 미래 잠재 서식지를 예측합니다. 모형은 현재 분포지에서 관찰되지 않은 2050년 고온 조건에서 적합도를 과대평가할 수 있으므로, 실제 분포지에서 관찰된 기온 한계(`data/Thermal_Limits.json`; Araújo et al., 2013)를 넘는 격자는 서식 불가지(적합도 0)로 처리합니다.
- **기온 한계값(`Thermal_Limits.json`)을 정한 방법($P_q$ = 출현 격자 분포의 $q$-백분위수)**:
  1. **전 지구 출현 격자 백분위수($P_{99}^{\text{GBIF}}, P_{01}^{\text{GBIF}}$)**: 화분매개자 233종·식물 274종에 대해 **GBIF API** 검증 좌표(종당 최대 900개)를 **10분(`10'`) WorldClim 2.1**(1970–2000; Fick & Hijmans, 2017) 고유 격자로 공간 중복 제거(thinning)한 뒤 산출한 `Bio5` 99백분위수($P_{99}^{\text{GBIF}}$) 및 `Bio6` 1백분위수($P_{01}^{\text{GBIF}}$)입니다.
  2. **국내 출현 격자 백분위수($P_{99}^{\text{KR}}, P_{01}^{\text{KR}}$)**: **제3~5차 전국자연환경조사(NES)** 1 km 출현 격자를 **기상청(KMA) CMIP6 1 km** 기준 기후장(2021–2025 5년 평균)에 중첩해 산출한 `Bio5` 99백분위수($P_{99}^{\text{KR}}$) 및 `Bio6` 1백분위수($P_{01}^{\text{KR}}$)입니다.
  3. **결합 규칙**: 국내 기록만 쓰면 온대·광역종의 고온 내성이 현재 남한 기온 상한(`Bio5 p99 = 33.5 °C`)에 인위적으로 절단되고, 반대로 1970–2000년 기준 `10'` WorldClim 2.1은 2021–2025년 1 km 기상청 격자보다 중앙값 기준 `3.30 °C` 낮아 단독 적용 시 현재 국내 출현지의 `5%` 이상을 오절단합니다. 따라서 하계 상한은 $T_{\max} = \max(P_{99}^{\text{KR}}, P_{99}^{\text{GBIF}})$로 결합하여 저온성 `FG03 뒤영벌속`(`33.41 °C`)과 고온 내성 `FG08 큰별노린재과`(`35.86 °C`)·`FG04 꿀벌속`(`41.53 °C`)을 구분합니다(`FG10`·`FG11`은 과 수준 GBIF $P_{99}$ 유지). 동계 하한($T_{\min}$)의 경우 온대 기능군은 GBIF 900좌표 표집 편향으로 강원 산지 출현지가 오절단되지 않도록 보수적인 목·과 수준 $P_{01}$(`-21.14 ~ -31.52 °C`)을 적용하고, 동계 저온에 실제로 제한되는 남방계 `FG08`만 경험적 $P_{01}$(`-13.96 °C`)을 사용합니다(Step 3 식물 기능군은 구성종 $P_{95}/P_{05}$ 또는 국내 출현 격자 $P_{99.5}+2\text{ °C}$ / $P_{0.5}-3\text{ °C}$ 적용). 이 값들은 실내 생리 실험의 치사온도($CT_{\max}/CT_{\min}$)가 아니라, 실제 분포지의 1 km·`10'` 격자 기후에서 관찰된 **분포 기후 한계(`realized macroclimatic bounds`)**입니다.

**`FG03 뒤영벌속`**으로 2050년 `BAU-SSP585` 서식지를 예측하고, 여름 최고기온 `33.41 °C` 이상 지역(`Bio5 >= 33.41 °C`)을 서식 불가지로 처리했을 때와 하지 않았을 때를 비교합니다. 각 격자는 2020년과 2050년 상태에 따라 `0: 계속 부적합`, `1: 서식지 소실`, `2: 서식지 유지(피난처)`, `3: 신규 서식 가능지` 네 유형으로 나눕니다.

In [ ]:
# [실습 2-B] FG03 뒤영벌속 2050년(BAU-SSP585) 서식지 예측: 여름 고온 한계 반영 여부 비교와 서식지 변화 4유형 지도
env_bio126 = load_env_scenario("env_2050_Biodiversity_SSP126")  # 2050 생태보전형 토지이용 + SSP1-2.6 기후 scenario 환경 raster 불러오기
env_bau585 = load_env_scenario("env_2050_BAU_SSP585")  # 2050 현 추세(BAU) 토지이용 + SSP5-8.5 고탄소 기후 scenario 환경 raster 불러오기

bombus_model = train_and_validate_sdm(bombus_ids, env_2020, bg_grid, beta_reg=1.5, use_hinge=True)  # 뒤영벌속 2020년 기준 SDM 적합 및 최적 임계치 산출
tau = bombus_model["threshold"]  # 교차검증 Max-TSS 기준 이진화 적합도 임계치(tau) 추출
t_max = bombus_tl["hot_limit_C"]  # 뒤영벌속의 여름 최고기온 상한(33.41 °C) 추출
t_min = bombus_tl["cold_limit_C"]  # 뒤영벌속의 겨울 최저기온 하한(보수적 설정)(-28.50 °C) 추출

# 1) 기온 한계 미반영(unmasked) vs. 반영(masked) 시 2020 -> 2050 서식 적합 면적 비교
hsi_2020_unmasked = bombus_model["hsi_2020_raw"]  # 기온 한계 반영 전 2020년 원시 서식 적합도 vector(길이 84,576) 추출
bin_2020_unmasked = hsi_2020_unmasked >= tau  # 임계치(tau) 이상인 2020년 적합 격자를 True로 이진화(mask 미적용)

hsi_2020_masked = np.where((env_2020["bio5"] >= t_max) | (env_2020["bio6"] <= t_min), 0.0, hsi_2020_unmasked)  # 2020년 기온 한계를 벗어난 격자를 서식 불가지(적합도 0)로 처리
bin_2020_masked = hsi_2020_masked >= tau  # 기온 한계 반영 후 2020년 적합 격자 이진화

X_2050_feat, _ = build_maxent_features(env_bau585["X_raw"], scaler=bombus_model["scaler"])  # 2050년 BAU-SSP585 환경변수를 2020년 훈련 scaler로 L/Q/H 변환
hsi_2050_unmasked = predict_suitability(bombus_model["params"], X_2050_feat, env_bau585["water_frac"])  # 기온 한계 반영 전 2050년 상대 서식 적합도 예측

exceed_thermal_585 = (env_bau585["bio5"] >= t_max) | (env_bau585["bio6"] <= t_min)  # 2050년 BAU-SSP585에서 기온 한계(특히 여름 최고기온 Bio5 >= 33.41 °C)를 벗어나는 격자 판별
hsi_2050_masked = np.where(exceed_thermal_585, 0.0, hsi_2050_unmasked)  # 기온 한계를 넘는 2050년 격자를 서식 불가지(적합도 0)로 처리

bin_2050_unmasked = hsi_2050_unmasked >= tau  # 기온 한계 미반영 시 2050년 적합 격자 이진화
bin_2050_masked = hsi_2050_masked >= tau  # 기온 한계 반영 시 2050년 적합 격자 이진화

chg_unmasked = (bin_2050_unmasked.sum() - bin_2020_unmasked.sum()) / bin_2020_unmasked.sum() * 100.0  # 기온 한계 미반영 시 2020 대비 2050 적합 면적 변화율(%) 계산
chg_masked = (bin_2050_masked.sum() - bin_2020_masked.sum()) / bin_2020_masked.sum() * 100.0  # 기온 한계 반영 시 2020 대비 2050 적합 면적 변화율(%) 계산

print(f"기온 한계 미반영: {bin_2020_unmasked.sum():,} -> {bin_2050_unmasked.sum():,} km² ({chg_unmasked:+.1f}%)")  # 기온 한계 미반영 시 적합 면적 변화 출력
print(f"여름 최고기온 {t_max} °C 이상 지역 제외: {bin_2020_masked.sum():,} -> {bin_2050_masked.sum():,} km² ({chg_masked:+.1f}%) | 2050년 여름 고온 한계 초과 격자: {exceed_thermal_585.sum():,}개")  # 기온 한계 반영 결과 출력

# 2) 서식지 변화 4유형 분류: 0=계속 부적합(Unsuitable), 1=서식지 소실(Loss), 2=서식지 유지(Stable), 3=신규 서식 가능지(Gain)
state_unmasked = np.zeros(len(hsi_2020_masked), dtype=np.int8)  # 기온 한계 미반영 시 변화 유형 배열을 0(현재·미래 모두 부적합)으로 초기화
state_unmasked[bin_2020_unmasked & (~bin_2050_unmasked)] = 1  # 2020 적합 -> 2050 부적합 격자에 1(소실, Loss) 부여
state_unmasked[bin_2020_unmasked & bin_2050_unmasked] = 2  # 2020 적합 -> 2050 적합 유지 격자에 2(유지, Stable) 부여
state_unmasked[(~bin_2020_unmasked) & bin_2050_unmasked] = 3  # 2020 부적합 -> 2050 신규 적합 격자에 3(신규 확장, Gain) 부여

state_masked = np.zeros(len(hsi_2020_masked), dtype=np.int8)  # 기온 한계 반영 전이 배열을 0(부적합)으로 초기화
state_masked[bin_2020_masked & (~bin_2050_masked)] = 1  # 기온 한계 반영 시 2020 적합 -> 2050 부적합 소실 격자(1) 지정
state_masked[bin_2020_masked & bin_2050_masked] = 2  # 기온 한계 반영 시 2020 적합 -> 2050 적합 유지 격자(2) 지정
state_masked[(~bin_2020_masked) & bin_2050_masked] = 3  # 기온 한계 반영 시 2020 부적합 -> 2050 신규 확장 격자(3) 지정

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0))  # 기온 한계 미반영·고온 초과 지역·반영 후 비교를 위한 1행 3열 지도 panel 생성
plot_4state_map(axes[0], state_unmasked, mask_2d, f"1. 고온 한계 미반영 ({chg_unmasked:+.1f}%)")  # 좌측 panel에 기온 한계 미반영 서식지 변화 4유형 지도 표시
axes[1].imshow(vec_to_2d(exceed_thermal_585.astype(float), mask_2d), cmap="Reds", vmin=0, vmax=1)  # 중앙 panel에 2050년 여름 고온 한계(33.41 °C) 초과 저지대 격자를 적색으로 표시
axes[1].set_title(f"2. 2050 여름 고온 한계 초과 (Bio5 >= {t_max} °C)", fontsize=10.2, fontweight="bold")  # 중앙 panel 제목 설정
axes[1].axis("off")  # 중앙 panel 축 눈금 숨김 처리
plot_4state_map(axes[2], state_masked, mask_2d, f"3. 고온 한계 반영 ({chg_masked:+.1f}%)")  # 우측 panel에 기온 한계 반영 후 서식지 변화 4유형 지도 표시
plt.tight_layout()  # 3개 지도 panel 여백 정렬
plt.show()  # 3열 비교 지도 출력

여름 최고기온 `33.41 °C` 이상 지역을 서식 불가지로 처리하면, 2050년 `BAU-SSP585`에서 내륙의 `39,589`개 저지대 격자(`46.8%`)가 이 한계를 넘기 때문에 `FG03 뒤영벌속`의 적합 서식지 감소 폭이 고온 한계 미반영 시 `43,396 -> 27,242 km² (-37.2%)`에서 **`43,201 -> 21,089 km² (-51.2%)`**로 심화됩니다. 다음 `cell`에서는 `project_to_scenario(...)` 함수로 **12개 기능군(`FG01`–`FG12`) 전체**의 `Biodiversity-SSP126`·`BAU-SSP585` 서식 면적 변화, 평균 고도 변화($\Delta\bar{z}$, 단위 `m`), 분포 중심 북상 거리($\Delta\bar{y}$, 단위 `km`)를 비교합니다.

In [ ]:
# [전체 비교 2-B] 12개 기능군의 2050년 서식 면적 변화, 산지 방향 고도 이동, 북상 거리 비교 (SSP1-2.6 vs. SSP5-8.5)
all_sub_proj_585 = {}  # Step 3 식물-화분매개자 중첩 분석에서 재사용할 기능군별 BAU-SSP585 예측 결과 저장 dict 초기화
summary_12 = []  # 12개 기능군 정량 요약 행을 담을 빈 list 초기화

for scode, skr, sen, oen in SUBGROUP_LIST:  # 12개 화분매개자 기능군을 순서대로 순회
    sids = pa_subgroups.loc[pa_subgroups["subgroup_kr"] == skr, "grid_id"].to_numpy()  # 현재 기능군의 1 km 출현 격자 ID 배열 추출
    tl_s = thermal_limits["subgroups"][skr]  # 현재 기능군의 실현 하계 상한(hot_limit_C) 및 동계 하한(cold_limit_C) 조회
    m_s = train_and_validate_sdm(sids, env_2020, bg_grid, beta_reg=1.5, use_hinge=True)  # 2020년 기준 3-겹 공간 block CV 및 SDM 적합 수행
    p126 = project_to_scenario(m_s, env_2020, env_bio126, bg_grid, tl_s["hot_limit_C"], tl_s["cold_limit_C"])  # 2050 Biodiversity-SSP126 scenario 서식지 예측
    p585 = project_to_scenario(m_s, env_2020, env_bau585, bg_grid, tl_s["hot_limit_C"], tl_s["cold_limit_C"])  # 2050 BAU-SSP585 scenario 서식지 예측
    all_sub_proj_585[skr] = p585  # Step 3 연계를 위해 국문 기능군명을 key로 BAU-SSP585 예측 결과 저장
    summary_12.append({  # 현재 기능군의 검증 및 2050 분포 변화 지표를 dict로 정리하여 추가
        "기능군_code": scode,  # 기능군 식별자(FG01~FG12)
        "상위_목": oen,  # 상위 곤충 목 영문명
        "기능군_국문명": skr,  # 기능군 국문명
        "공간CV_AUC": m_s["cv_metrics"]["auc"],  # 50 km 공간 block 교차검증 AUC
        "하계상한_C": tl_s["hot_limit_C"],  # 적용된 Bio5 상한 온도(°C)
        "적합면적_2020_km2": p585["area_2020_km2"],  # 2020년 기준 적합 서식지 면적(km²)
        "SSP126_변화율_pct": p126["pct_change"],  # 2050 Biodiversity-SSP126 적합 면적 변화율(%)
        "SSP585_변화율_pct": p585["pct_change"],  # 2050 BAU-SSP585 적합 면적 변화율(%)
        "고도이동_585_m": p585["elev_shift_m"],  # 적합 격자 평균 해발고도의 수직 변화량(m a.s.l.; 주로 저지대 소실에 의해 상승)
        "북상이동_585_km": p585["north_shift_km"],  # 적합 격자 분포 중심(centroid)의 수평 북상 이동 거리(km)
    })  # 현재 기능군 결과 저장 완료

pd.DataFrame(summary_12)  # 12개 기능군 전체의 2050 scenario별 분포 변화 요약표 출력

`BAU-SSP585` 아래에서 저온 적응 기능군(`FG12 하늘소과 -55.7%`, `FG03 뒤영벌속 -51.2%`, `FG09 호랑나비과 -50.9%`, `FG02 재니등에과 -50.7%`, `FG06 단독성 야생벌류 -48.3%`, `FG07 온대 노린재류 -46.2%`)은 대규모 서식지 축소와 평균 고도 상승(`+67.6 ~ +139.1 m`)을 겪는 반면, 남해안 아열대성 `FG08 큰별노린재과`는 고도 변화(`-1.2 m`) 없이 해안·저지대를 따라 북상(`+24.3 km`)하며 **`+87.9%`** 확장합니다.

## Step 3. 식물–화분매개자 공존 서식지 변화와 공간 불일치 (`Slides 25–29`)
분류군별 단일 SDM만으로는 먹이식물과의 공간 공존 여부를 평가할 수 없으므로(계절현상 불일치 하에서 비생물적 SDM 위험도와 식물 2차 절멸 위험 간 상관은 $r < 0.3$에 불과; Peng et al., 2025), 화분매개자와 밀원·기주식물의 서식 적합지가 얼마나 겹치는지(**Schoener's $D$**; Warren et al., 2008), 그리고 2020년 공존 격자의 2050년 변화 4유형(Schweiger et al., 2008; Schleuning et al., 2016; Briggs & Anderson, 2026)을 정량화합니다.

**`FG03 뒤영벌속` $\times$ `봄철·목본 핵심 밀원군`** 쌍(이른 봄 산림·임연부 목본 밀원과 뒤영벌)의 서식지 중첩도와 공존 서식지 변화 유형을 계산합니다.

In [ ]:
# [실습 3] FG03 뒤영벌속과 봄철 목본 밀원식물: 서식지 겹침(Schoener's D)과 공존 서식지의 2050년 변화 유형 계산
plant_focal_kr = "봄철·목본 핵심 밀원군"  # 뒤영벌속의 핵심 봄철 밀원 partner인 식물 기능군 국문명 지정
plant_ids = pa_plants.loc[pa_plants["plant_group_kr"] == plant_focal_kr, "grid_id"].to_numpy()  # 해당 식물 기능군의 1 km 출현 격자 ID 배열 추출
plant_tl = thermal_limits["plants"][plant_focal_kr]  # 해당 식물 기능군의 분포 기온 상·하한 임계치 조회

# 1) 밀원식물 기능군 SDM을 적합하고 2050 BAU-SSP585 서식지 예측
plant_model = train_and_validate_sdm(plant_ids, env_2020, bg_grid, use_target_group_bg=False, beta_reg=1.5, use_hinge=True)  # 봄철·목본 밀원식물 SDM 적합
spring_plant_585 = project_to_scenario(plant_model, env_2020, env_bau585, bg_grid, plant_tl["hot_limit_C"], plant_tl["cold_limit_C"])  # 밀원식물 2050 BAU-SSP585 서식지 예측
bombus_585 = all_sub_proj_585["뒤영벌속"]  # 앞서 Step 2-B에서 산출한 뒤영벌속의 BAU-SSP585 예측 결과 불러오기

# 2) 지리적 적합도 중첩 지수 Schoener's D = 1 - 0.5 * sum(|p_poll_norm - p_plant_norm|) 수식 직접 계산
p_poll_20 = bombus_585["hsi_2020"] / bombus_585["hsi_2020"].sum()  # 2020년 뒤영벌속 적합도 합계가 1이 되도록 확률분포 정규화
p_plnt_20 = spring_plant_585["hsi_2020"] / spring_plant_585["hsi_2020"].sum()  # 2020년 봄철 밀원식물 적합도 합계가 1이 되도록 확률분포 정규화
d_2020 = 1.0 - 0.5 * np.sum(np.abs(p_poll_20 - p_plnt_20))  # 2020년 기준 두 분류군 간 지리적 서식 적합도 중첩도(Schoener's D) 산출

p_poll_50 = bombus_585["hsi_2050"] / bombus_585["hsi_2050"].sum()  # 2050년 뒤영벌속 적합도 vector를 합계 1로 정규화
p_plnt_50 = spring_plant_585["hsi_2050"] / spring_plant_585["hsi_2050"].sum()  # 2050년 봄철 밀원식물 적합도 vector를 합계 1로 정규화
d_2050 = 1.0 - 0.5 * np.sum(np.abs(p_poll_50 - p_plnt_50))  # 2050년 BAU-SSP585 기준 지리적 서식 적합도 중첩도(Schoener's D) 산출

# 3) 2020년에 둘이 함께 살던 격자(co_2020)가 2050년에 어떻게 바뀌는지 4유형으로 분류
v20, v50 = bombus_585["bin_2020"], bombus_585["bin_2050"]  # 뒤영벌속의 2020년 및 2050년 이진 적합 격자(True/False) 추출
p20, p50 = spring_plant_585["bin_2020"], spring_plant_585["bin_2050"]  # 봄철 밀원식물의 2020년 및 2050년 이진 적합 격자(True/False) 추출
co_2020 = v20 & p20  # 2020년에 화분매개자와 밀원식물이 모두 적합한 2020년 공존 격자 판별
co_2050 = v50 & p50  # 2050년에 화분매개자와 밀원식물이 모두 적합한 미래 공존 격자 판별

mismatch_1d = np.zeros(len(v20), dtype=np.int8)  # 전 격자 상태값을 0(2020년 비공존 배경)으로 초기화
mismatch_1d[co_2020 & v50 & p50] = 1  # 1: 공존 유지(Co-occurrence Persist — 기후·먹이원 동시 유지 피난처)
mismatch_1d[co_2020 & v50 & (~p50)] = 2  # 2: 식물만 소실(Plant Lost — 화분매개자의 먹이·밀원 결핍)
mismatch_1d[co_2020 & (~v50) & p50] = 3  # 3: 화분매개자만 소실(Pollinator Lost — 잔존 식물의 수분 결핍)
mismatch_1d[co_2020 & (~v50) & (~p50)] = 4  # 4: 양측 동시 소실(Both Lost — 저지대 동반 국지 절멸)

n_co20 = int(co_2020.sum())  # 2020년 기준 공존 격자 총수(km²) 계산
co_chg = (int(co_2050.sum()) - n_co20) / n_co20 * 100.0  # 2020년 대비 2050년 총 공존 적합 면적 증감률(%) 계산
print(f"[FG03 뒤영벌속 x 봄철·목본 밀원군] Schoener's D: {d_2020:.2f} (2020) -> {d_2050:.2f} (2050) | 공존 면적 변화율: {co_chg:+.1f}%")  # 중첩 지수 및 공존 면적 변화 출력
print(f"2020년 공존 {n_co20:,}개 격자의 2050년 변화 -> 1.공존유지: {(mismatch_1d == 1).sum() / n_co20 * 100:.1f}%, 2.식물소실: {(mismatch_1d == 2).sum() / n_co20 * 100:.1f}%, 3.매개자소실: {(mismatch_1d == 3).sum() / n_co20 * 100:.1f}%, 4.동시소실: {(mismatch_1d == 4).sum() / n_co20 * 100:.1f}%")  # 서식지 변화 4유형 구성비 출력

# 4) 공존 서식지 변화 4유형 지도 시각화
cmap_mm = ListedColormap([GFO_COLORS["stone"], GFO_COLORS["pine"], GFO_COLORS["ochre"], GFO_COLORS["slate"], GFO_COLORS["terracotta"]])  # 0~4 상태별 5색 이산 colormap 생성
norm_mm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap_mm.N)  # 정수 상태값(0,1,2,3,4)이 각 색상에 정확히 대응되도록 경계 설정

fig, ax = plt.subplots(figsize=(5.4, 5.2))  # 단일 공간 불일치 지도 figure 생성
im = ax.imshow(vec_to_2d(mismatch_1d, mask_2d), cmap=cmap_mm, norm=norm_mm)  # 1차원 불일치 상태 vector를 2차원 지도로 복원하여 표시
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4], fraction=0.046, pad=0.03)  # 우측에 5개 범주 눈금을 가진 colorbar 부착
cbar.ax.set_yticklabels(["0: 2020년 비공존", "1: 공존 유지", "2: 식물 소실", "3: 화분매개자 소실", "4: 동시 소실"])  # colorbar 눈금에 국문 상태 label 지정
ax.set_title("FG03 뒤영벌속 × 봄철·목본 밀원군 (2050 BAU-SSP585)", fontsize=10.2, fontweight="bold")  # 지도 제목 설정
ax.axis("off")  # 지도 테두리 눈금 숨김 처리
plt.tight_layout()  # 여백 자동 조정
plt.show()  # 공간 불일치 지도 출력

`BAU-SSP585` 아래에서 `봄철·목본 핵심 밀원군`은 저지대에서도 비교적 넓게 잔존하는 반면 `FG03 뒤영벌속`이 고온 한계(`Bio5 >= 33.41 °C`)로 산지로 후퇴하면서, 두 분류군 간 `Schoener's D`가 `0.84`에서 `0.64`로 하락하고 2020년 공존 격자의 **`23.1%`**가 식물만 남고 뒤영벌이 사라지는 **수분 공백(`3: 화분매개자 소실`)** 상태로 전환됩니다(2020년 비공존 격자 `36.9%`까지 포함한 2050년 전체 적합 봄철 목본 격자 기준으로는 `54.0%`에서 뒤영벌 부재). 다음 `cell`에서는 **5개 식물 기능군**과 **8개 핵심 상호작용 쌍**에 대해 어느 쌍에서 먹이 부족(식물 소실)과 수분 공백(화분매개자 소실)이 두드러지는지 비교합니다.

In [ ]:
# [전체 비교 3] 나머지 식물 기능군 SDM을 적합하고 8개 식물-화분매개자 쌍의 공존 서식지 변화 비교
plant_proj_585 = {plant_focal_kr: spring_plant_585}  # 이미 계산한 봄철·목본 핵심 밀원군 예측 결과를 dict에 먼저 등록
for pkr, pmeta in thermal_limits["plants"].items():  # Thermal_Limits.json에 정의된 5개 식물 기능군을 순회
    if pkr in plant_proj_585:  # 이미 적합·예측이 끝난 봄철·목본 밀원군이면 중복 계산 건너뛰기
        continue  # 다음 식물 기능군으로 이동
    pids = pa_plants.loc[pa_plants["plant_group_kr"] == pkr, "grid_id"].to_numpy()  # 현재 식물 기능군의 1 km 출현 격자 ID 추출
    pm = train_and_validate_sdm(pids, env_2020, bg_grid, use_target_group_bg=False, beta_reg=1.5, use_hinge=True)  # 식물 기능군 SDM 적합(전국 배경 10,000격자 기준)
    plant_proj_585[pkr] = project_to_scenario(pm, env_2020, env_bau585, bg_grid, pmeta["hot_limit_C"], pmeta["cold_limit_C"])  # 2050 BAU-SSP585 scenario 예측 결과 저장

pairs = [  # 생태적 수분·기주 관계에 기반한 8개 핵심 (상호작용 쌍 명칭, 화분매개자 예측, 식물 예측) list 정의
    ("FG03 뒤영벌속 × 봄철·목본 밀원군", all_sub_proj_585["뒤영벌속"], plant_proj_585["봄철·목본 핵심 밀원군"]),  # 봄철 산림·임연부 목본 개화기 핵심 수분 쌍
    ("FG06 단독성 야생벌류 × 봄철·목본 밀원군", all_sub_proj_585["단독성 야생벌류"], plant_proj_585["봄철·목본 핵심 밀원군"]),  # 봄철 단독성 야생벌과 목본 밀원 쌍
    ("FG01 꽃등에과 × 초지 국화과·산형과 밀원군", all_sub_proj_585["꽃등에과"], plant_proj_585["초지 국화과·산형과 밀원군"]),  # 개방 초지 복총산형·두상화서 밀원 쌍
    ("FG02 재니등에과 × 초지·습초지 식물군", all_sub_proj_585["재니등에과"], plant_proj_585["초지·습초지 식물군"]),  # 건조·개방 초지 재니등에와 초지 식물 쌍
    ("FG09 호랑나비과 × 산림성 식물군", all_sub_proj_585["호랑나비과"], plant_proj_585["산림성 식물군"]),  # 산림 내부 호랑나비류와 산림 기주·밀원 쌍
    ("FG10 네발나비과 × 콩과·제비꽃 기주·밀원군", all_sub_proj_585["네발나비과"], plant_proj_585["콩과·제비꽃 기주·밀원군"]),  # 네발나비류 유충 핵심 기주(제비꽃·콩과) 쌍
    ("FG07 온대 노린재류 × 초지·습초지 식물군", all_sub_proj_585["온대 노린재류"], plant_proj_585["초지·습초지 식물군"]),  # 초지 식물 섭식·방문 온대 노린재 쌍
    ("FG12 하늘소과 × 산림성 식물군", all_sub_proj_585["하늘소과"], plant_proj_585["산림성 식물군"]),  # 산림 고사목·수관 의존 꽃하늘소와 산림 식물 쌍
]  # 8개 상호작용 쌍 list 정의 종료

ov_rows = []  # 8개 쌍의 공존 분석 지표를 저장할 list 초기화
for label, vp, pp in pairs:  # 8개 식물-화분매개자 쌍을 순서대로 순회
    ov = analyze_bipartite_overlap(vp, pp)  # Schoener's D, 공존 면적 증감률 및 공존 변화 4유형 비율 계산
    ov_rows.append({  # 분석 결과를 표 출력용 dict로 구성하여 추가
        "상호작용_쌍": label,  # 식물-화분매개자 상호작용 쌍 이름
        "공존면적_변화율_2050_pct": ov["cooccur_change_pct"],  # 2020 대비 2050 공존 적합 격자 증감률(%)
        "Schoener_D_20to50": f"{ov['schoeners_d_2020']:.2f} -> {ov['schoeners_d_2050']:.2f}",  # 2020년 -> 2050년 지리적 적합도 중첩도 변화
        "공존유지_pct": ov["state_1_persist_pct"],  # 2020 공존 격자 중 양측 모두 유지되는 비율(%)
        "식물소실_먹이결핍_pct": ov["state_2_plant_lost_pct"],  # 2020 공존 격자 중 식물만 소실되는 비율(%)
        "화분매개자소실_수분공백_pct": ov["state_3_poll_lost_pct"],  # 2020 공존 격자 중 화분매개자만 소실되는 비율(%)
        "동시소실_pct": ov["state_4_both_lost_pct"],  # 2020 공존 격자 중 양측 모두 소실되는 비율(%)
    })  # 단일 상호작용 쌍 요약 추가 완료

pd.DataFrame(ov_rows)  # 8개 상호작용 쌍 전체의 공존 변화 요약표 출력

`FG01 꽃등에과`는 단일 분류군 기준 `-30.9%` 감소하지만 상호작용 `partner`인 `초지 국화과·산형과 밀원군`(`-23.9%`)의 축소가 겹치면서 공존 적합지가 **`-34.2%`**로 더 크게 줄어들며(`식물 소실` `17.9%`), 산지 후퇴 폭이 큰 `FG03 뒤영벌속`(`23.1%`), `FG09 호랑나비과`(`22.9%`), `FG12 하늘소과`(`21.0%`) 쌍에서는 2020년 공존지의 5분의 1 이상에서 화분매개자 소실에 따른 수분 공백이 발생합니다(`Slide 28`).

이어서 개별 기능군 쌍을 넘어 **육상식물 `274`종 $\times$ 화분매개자 `139`종 적층 종풍부도(`SSDM`)** 전체에서 산출한 전국·고도대별 4유형 공간 불일치 합성표(`outputs/step3_community_regimes_summary.csv`; `Slide 29` 및 `Slide 40`)를 확인합니다(기저 종풍부도 $S_{2020} \ge 5$ 유효 격자 `76,575`개, 내륙의 `90.5%`; 양측 모두 종풍부도 변화 $\Delta S \ge -20\%$ 유지 시 공존 피난처로 분류).

In [ ]:
# [전체 비교 3-B] 274종 식물 x 139종 화분매개자 군집 수준 전국·고도대별 공존 피난처 및 불일치 비율 확인 (Slide 29)
comm_regimes = pd.read_csv(os.path.join(WORKSHOP_ROOT, "outputs", "step3_community_regimes_summary.csv"))  # 사전 집계된 군집 수준 4유형 공간 불일치 요약 CSV file 읽기
comm_regimes.rename(columns={  # 영문 열 이름을 국문 의미가 드러나는 열 이름으로 변환하여 출력
    "Scope": "분석_범위_및_scenario",  # 전국 scenario 비교 또는 BAU-SSP585 해발고도 구간 구분
    "Valid_Cells_S_ge_5": "유효격자수_S5이상",  # 2020년 기준 최소 5종 이상 서식하는 유효 평가 1 km 격자 수
    "Low_Baseline_lt5_Cells": "저밀도제외_격자수",  # 기저 종풍부도 5종 미만으로 비율 산정에서 제외한 격자 수
    "Refugia_pct": "공존피난처_유지_pct",  # 식물과 화분매개자 군집이 함께 유지되는 피난처 비율(%)
    "Pollinator_Deficit_pct": "화분매개자결핍_pct",  # 식물은 유지되나 화분매개자 군집만 급감하는 비율(%)
    "Floral_Host_Deficit_pct": "밀원기주식물결핍_pct",  # 화분매개자 기후는 유지되나 먹이·기주식물이 급감하는 비율(%)
    "Dual_Loss_pct": "양측동반손실_pct",  # 식물과 화분매개자 군집이 동시에 급감하는 비율(%)
})  # 국문 열 이름이 적용된 군집 공간 불일치 요약 DataFrame 출력

기후 완화와 서식지 보전을 결합한 **`Biodiversity-SSP126`**은 전국 유효 서식 격자의 **`29.9%`**를 식물–화분매개자 공존 피난처로 유지하여 **`BAU-SSP585`**(**`15.2%`**) 대비 약 두 배의 상호작용 보전 효과를 보이며, `BAU-SSP585`에서는 해발 `200–500 m` 산록부의 **`51.1%`**가 밀원·기주식물 결핍 구간으로 전환됩니다(`Slide 29`, `Slide 40`).

## Step 4. 군집 종조성 변화: 종별 예측을 쌓는 방법(SSDM)과 지점 간 종조성 차이를 직접 모형화하는 방법(GDM) (`Slides 30–39`)
- **종별 예측을 쌓아 군집 만들기(적층 종분포모형, `SSDM` — *"개별 예측 후 군집 적층 [Predict first, assemble later]"*; Ferrier & Guisan, 2006; Dubuis et al., 2011; Calabrese et al., 2014; Zurell et al., 2020)**: 개별 종·기능군의 서식 적합도 $p_{ik}$를 먼저 추정한 뒤 적층하여 **종풍부도 변화($\Delta S$)**, 기온 한계를 넘는 저지대에서의 종 소실, 그리고 **종조성 변화의 두 성분(종 교체 vs. 종 감소; Baselga, 2010)**($\beta_{\text{sor}} = \beta_{\text{sim}} + \beta_{\text{sne}}$)을 정량화합니다.
- **지점 간 종조성 차이를 직접 모형화(일반화 비유사도 모형, `GDM` — *"군집 집계 후 비유사도 예측 [Assemble first, predict later]"*; Ferrier et al., 2007; Mokany et al., 2022)**: 지점 쌍 간 종조성 비유사도($d_{ij} = 1 - e^{-\eta_{ij}}$)를 3-매듭 단조 3차 `I-spline`으로 직접 모형화하여 **환경 구배의 어느 구간에서 종조성이 급격히 바뀌는지(비선형 변화 구간)**와 희귀종을 포함한 군집 전체의 변화를 포착합니다.
- **국가 단위 기후 $\times$ 토지이용 `scenario` 결합 연구와의 비교**: 영국의 **Cooke et al. (2026, *Nature Communications*, 17, 2704)**은 표준화된 반복 `monitoring` 자료(식물 1,002종, 나비 56종, 조류 219종; 모형 설명력 31–56%)로 적합한 1 km GDM을 행위자 기반 토지이용 모형 **CRAFTY-GB**(Brown et al., 2022)와 결합했습니다. 반면 국내 전국자연환경조사의 단회 방문 곤충 출현 기록(`PO`)은 지점 쌍의 `45.5%`가 공유종이 하나도 없어 비유사도가 최댓값($d_{ij} = 1$)에 머무르므로 원시 곤충 GDM 설명력이 `0.81%`(식물 $S \ge 5$ 관측 행렬은 `37.04%`)에 그치므로, 본 `workshop`에서는 **Dyna-CLUE (100 m)** 토지피복 변화 예측을 139종 **SSDM**과 결합하고 **SSDM 예측 군집에 GDM**(모형 설명력 `73.70%`) 및 Baselga $\beta_{\text{sim}} / \beta_{\text{sne}}$ 분할을 함께 적용합니다.
- **변수 중요도 정의 차이(`SDM` vs. `GDM`)**: SDM은 표준화 `L/Q/H` 계수 절댓값 합($\sum_m |\hat{\beta}_{km}|$) 비중으로 국지 서식 민감도를 요약하는 반면, GDM은 **`I-spline` 최대 높이**($H_k = \sum_{m=1}^3 a_{km}$; Ferrier et al., 2007)로, 해당 환경 구배의 한쪽 끝에서 다른 쪽 끝까지 가는 동안 누적되는 종조성 변화량을 나타냅니다.
- **Baselga (2010) 분할 수식**: 전체 Sørensen 비유사도 $\beta_{\text{sor}} = (b + c)/(2a + b + c)$를 **종 교체(`Simpson turnover`; 빠진 종의 자리를 다른 종이 채움)** $\beta_{\text{sim}} = \min(b, c)/[a + \min(b, c)]$(1:1 종 치환)와 **종 감소에 따른 차이(`nestedness-resultant dissimilarity`; 새 종 유입 없이 일부 종만 빠져 한쪽 군집이 다른 쪽의 부분집합이 됨)** $\beta_{\text{sne}} = \beta_{\text{sor}} - \beta_{\text{sim}}$(대체 종 유입 없는 종 감소; Carvalho et al., 2012의 $\beta_{\text{rich}}$ 논의 참조)로 분해합니다.
- **방법론적 한계**: SSDM과 GDM 모두 **공간–시간 치환(`space-for-time substitution`; 현재의 지역 간 차이로 미래의 시간 변화를 추정)**과, 종이 새 적합지로 제약 없이 이동한다는 가정(무제한 산포)을 가정하므로 잠재 적합도가 곧 실현 미래 군집을 의미하지 않으며(Blois et al., 2013; Fitzpatrick & Keller, 2015), 개체수·경쟁·계절현상 불일치(Peng et al., 2025)를 직접 반영하지 못하므로 기후 외삽 진단(**MESS** Elith et al., 2010; **ExDet** Mesgaran et al., 2014)과 병행 해석해야 합니다.

![개념 비교: SSDM(개별 종 적층) vs. GDM(지점 쌍 비유사도 모형화)](figures/comics/concept_sdm_vs_gdm.png)

![종조성 변화를 보는 두 관점: 환경 구배 I-spline과 Baselga 집합 분할](figures/comics/concept_turnover_explained.png)

### Step 4 단일 사례: 두 조사 지점의 종조성 차이 나누기(종 교체 vs. 종 감소)와 기온에 따른 종조성 변화곡선
관측 육상식물 출현 행렬(`GDM_Community_Benchmark.npz`)에서 두 조사 격자(`Site 0`과 `Site 15`)를 선택해 공유 종수 `a`와 고유 종수 `b`, `c`를 직접 세고 **Baselga의 $\beta_{\text{sor}} = \beta_{\text{sim}} + \beta_{\text{sne}}$** 식으로 종조성 차이가 종 교체와 종 감소로 정확히 나뉘는지 확인합니다.

In [ ]:
# [실습 4-A] 두 실제 조사 지점(Site 0 vs. Site 15)에서 Sørensen 비유사도 및 Baselga (2010) 분할을 단계별로 직접 계산
bm = np.load(os.path.join(DATA_DIR, "GDM_Community_Benchmark.npz"))  # 사전 구축된 군집 benchmark 행렬(식물·곤충 관측치 및 139종 SSDM 표본) 불러오기
plant_s5_ids, plant_s5_mat = bm["plant_obs_s5_ids"], bm["plant_obs_s5_mat"]  # 5종 이상 기록된 육상식물 조사 격자 ID와 지점x종(274종) 출현 행렬 추출

site_i = plant_s5_mat[0] > 0  # 첫 번째 조사 지점(Site 0)에서 출현한 식물 종 위치를 True로 변환
site_j = plant_s5_mat[15] > 0  # 비교 대상 조사 지점(Site 15)에서 출현한 식물 종 위치를 True로 변환

a = int(np.sum(site_i & site_j))  # a: 두 지점(Site 0, Site 15) 모두에 공통으로 출현한 공유 종수 계산
b = int(np.sum(site_i & (~site_j)))  # b: Site 0에만 출현하고 Site 15에는 없는 고유 종수 계산
c = int(np.sum((~site_i) & site_j))  # c: Site 15에만 출현하고 Site 0에는 없는 고유 종수 계산

beta_sor = (b + c) / (2 * a + b + c)  # 전체 Sørensen 종조성 비유사도(beta_sor, 0=완전 동일 ~ 1=공유종 전무) 계산
beta_sim = min(b, c) / (a + min(b, c))  # 빠진 종의 자리를 다른 종이 채운 몫인 종 교체 성분(Simpson, beta_sim) 계산
beta_sne = beta_sor - beta_sim  # 종수 차이에서 생기는 종 감소 성분(중첩성, beta_sne) 계산

print(f"Site 0 (종수 S = {int(site_i.sum())}) vs. Site 15 (종수 S = {int(site_j.sum())}): 공유종 a = {a}, Site0 고유종 b = {b}, Site15 고유종 c = {c}")  # a, b, c 집계 출력
print(f"전체 Sørensen (beta_sor) = {beta_sor:.3f}  ==  종 교체 (beta_sim) {beta_sim:.3f} + 종 감소 (beta_sne) {beta_sne:.3f}")  # 분할 합 일치 확인 출력

두 지점(`Site 0` 종수 `S = 9`, `Site 15` 종수 `S = 47`) 간 공유종(`a = 3`)과 고유종(`b = 6`, `c = 44`)만으로 전체 종조성 차이($\beta_{\text{sor}} = 0.893$)가 순수 종 교체($\beta_{\text{sim}} = 0.667$)와 종수 차이에 따른 종 감소 성분($\beta_{\text{sne}} = 0.226$)의 합으로 정확히 분해됨을 확인했습니다.

다음 `cell`에서는 139종 화분매개자 SSDM 군집에 GDM을 적합한 뒤, 비음수 제약(환경 차이 증가 시 종조성 차이가 감소하지 않도록 함) `spline` 가중치 $(a_1, a_2, a_3 \ge 0)$로부터 연평균기온(`bioclim1`)에 따른 **종조성 변화곡선(3-매듭 단조 `I-spline`)**($H_k = a_1 + a_2 + a_3$)을 계산합니다. 기울기가 가파른 구간일수록 같은 기온 차이에서 종조성 차이가 큽니다.

In [ ]:
# [실습 4-B] 139종 화분매개자 SSDM 군집에 GDM을 적합하고 연평균기온(bioclim1)에 따른 종조성 변화곡선(I-spline) 계산
X_2020 = env_2020["X_raw"]  # 2020년 기준 22개 환경 예측변수 원시 행렬(84,576 x 22) 참조
coords_km = bg_grid[["x_5186", "y_5186"]].to_numpy(dtype=np.float64) / 1000.0  # EPSG:5186 평면직각좌표(m)를 km 단위 평면 좌표로 변환
ssdm_ids, poll_ssdm_mat = bm["ssdm_sample_ids"], bm["poll_ssdm_sample_mat"].astype(np.float32)  # SSDM 표본 격자 ID와 139종 연속 적합도 행렬 추출

gdm_ssdm = fit_gdm_ispline(poll_ssdm_mat, X_2020[ssdm_ids], coords_km[ssdm_ids], DEFAULT_DRIVERS, metric="braycurtis", max_pairs=25000)  # 25,000개 지점 쌍에 대해 단조 I-spline GDM 적합

# bioclim1(연평균기온)의 3개 매듭(최솟값, 중앙값, 최댓값)과 비음수 spline 계수(a1, a2, a3 >= 0) 추출
vcode = "bioclim1"  # I-spline 곡선을 직접 복원할 환경변수 code 지정
knots_b1 = gdm_ssdm["knots"][vcode]  # bioclim1의 3개 분위수 매듭 위치(min, median, max) 추출
coefs_b1 = gdm_ssdm["var_coefs_3"][vcode]  # NNLS(비음수 최소제곱)로 추정된 3개 기저 가중치 vector(a1, a2, a3) 추출
height_b1 = float(np.sum(coefs_b1))  # 기온 구배 전체에서의 누적 종조성 변화량인 최대 spline 높이(H_k = a1 + a2 + a3) 계산

x_grid = np.linspace(knots_b1[0], knots_b1[2], 200)  # 최솟값 매듭부터 최댓값 매듭까지 등간격 200개 온도 평가 지점 생성
B_basis = eval_ispline_3(x_grid, knots_b1)  # 200개 온도 지점에서 (200 x 3) 크기의 단조 3차 I-spline 기저 행렬 계산
I_curve = B_basis @ coefs_b1  # 기저 행렬과 가중치 vector의 행렬곱으로 부분 생태적 거리 곡선 I_k(x) = a1*B1 + a2*B2 + a3*B3 산출

print(f"[GDM bioclim1] 3-매듭 위치 (최소, 중앙, 최대): ({knots_b1[0]:.1f}, {knots_b1[1]:.1f}, {knots_b1[2]:.1f}) °C")  # 3개 spline 매듭 온도 출력
print(f"비음수 I-spline 가중치 (a1, a2, a3): {np.round(coefs_b1, 3)} -> 최대 높이 H_k = {height_b1:.3f} (환경에 따른 전체 종조성 변화의 {gdm_ssdm['var_env_imp'][vcode]:.1f}%)")  # 가중치 및 중요도 출력

fig, ax = plt.subplots(figsize=(5.8, 3.4))  # I-spline 곡선 시각화를 위한 단일 figure 생성
ax.plot(x_grid, I_curve, lw=2.5, color=GFO_COLORS["terracotta"], label=f"I(bioclim1), 최대 높이 H = {height_b1:.2f}")  # 단조 증가하는 부분 생태적 거리 곡선 그리기
for kn in knots_b1:  # 최소·중앙·최대 3개 매듭 위치를 순회
    ax.axvline(kn, color="gray", ls=":", lw=0.9)  # 각 매듭 온도에 회색 세로 점선 표시
ax.set_xlabel("연평균기온 (bioclim1, °C)")  # 가로축 label 설정
ax.set_ylabel("누적 종조성 변화 I(x)")  # 세로축 label 설정
ax.set_title("연평균기온에 따른 종조성 변화곡선 (GDM I-spline)", fontweight="bold")  # plot 제목 설정
ax.legend()  # 범례 표시
plt.tight_layout()  # plot 여백 자동 조정
plt.show()  # I-spline plot 출력

`bioclim1`(연평균기온)의 3개 분위수 매듭 `(6.1, 12.6, 16.5) °C`에서 최대 `spline` 높이는 $H_k = 0.608$(환경에 따른 전체 종조성 변화의 `10.7%`)이며, 기울기가 가파른 저온~중온 전이 구간에서 부분 생태적 거리 $I(x)$가 가장 크게 증가합니다.

### Step 4 전체 비교: 조사 자료의 완전성이 군집 분석에 미치는 영향과 2050년 전국 군집 변화 지도
단회 방문 위주의 불완전한 곤충 출현 기록이 군집 분석에 미치는 영향을 비교하기 위해 **4가지 군집 자료 체계**에서 GDM 설명력을 비교한 뒤, 2050년(`BAU-SSP585`) 전국 **GDM 종조성 변화** 지도와 **SSDM Baselga 종 교체($\beta_{\text{sim}}$)·종 감소($\beta_{\text{sne}}$)** 지도를 나란히 비교합니다.

In [ ]:
# [전체 비교 4-A] 조사 완전성과 분류군 특성이 다른 4가지 군집 자료의 GDM 설명력 비교
poll_all_ids, poll_all_mat = bm["poll_obs_all_ids"], bm["poll_obs_all_mat"]  # 1종 이상 기록된 전체 화분매개자 조사 격자 ID 및 관측 출현 행렬 추출
mask_s10 = np.sum(poll_all_mat > 0, axis=1) >= 10  # 격자당 최소 10종 이상 기록된 고밀도 조사 격자만 선별하는 boolean mask 생성

gdm_raw = fit_gdm_ispline(poll_all_mat, X_2020[poll_all_ids], coords_km[poll_all_ids], DEFAULT_DRIVERS, metric="sorensen", max_pairs=25000)  # 체계 1: 원시 곤충 PO 행렬(S>=1) GDM 적합
gdm_s10 = fit_gdm_ispline(poll_all_mat[mask_s10], X_2020[poll_all_ids[mask_s10]], coords_km[poll_all_ids[mask_s10]], DEFAULT_DRIVERS, metric="sorensen", use_weights=True, max_pairs=25000)  # 체계 2: 고밀도 곤충 PO 행렬(S>=10, 종수 가중) GDM 적합
gdm_plant = fit_gdm_ispline(plant_s5_mat, X_2020[plant_s5_ids], coords_km[plant_s5_ids], DEFAULT_DRIVERS, metric="sorensen", use_weights=True, max_pairs=25000)  # 체계 3: 관측 육상식물 PO 행렬(S>=5, 종수 가중) GDM 적합

pd.DataFrame([  # 4가지 GDM 모형 체계의 표본 지점 수, 공유종이 없는 지점 쌍 비율(d=1), 모형 설명력(%), 상관계수(r) 비교표 생성
    {"모형_체계": "1. 원시 곤충 출현 행렬 (1 km PO, S>=1)", "조사_격자수": gdm_raw["n_sites"], "공유종없는_쌍_비율_d1_pct": gdm_raw["pct_saturated_d1"], "모형_설명력_pct": gdm_raw["pct_dev_explained"], "상관계수_r": gdm_raw["pearson_r"]},  # 체계 1 성능 요약
    {"모형_체계": "2. 선별 곤충 출현 행렬 (1 km PO, S>=10)", "조사_격자수": gdm_s10["n_sites"], "공유종없는_쌍_비율_d1_pct": gdm_s10["pct_saturated_d1"], "모형_설명력_pct": gdm_s10["pct_dev_explained"], "상관계수_r": gdm_s10["pearson_r"]},  # 체계 2 성능 요약
    {"모형_체계": "3. 관측 육상식물 출현 행렬 (1 km PO, S>=5)", "조사_격자수": gdm_plant["n_sites"], "공유종없는_쌍_비율_d1_pct": gdm_plant["pct_saturated_d1"], "모형_설명력_pct": gdm_plant["pct_dev_explained"], "상관계수_r": gdm_plant["pearson_r"]},  # 체계 3 성능 요약
    {"모형_체계": "4. 화분매개자 SSDM 적층 군집 (139종)", "조사_격자수": gdm_ssdm["n_sites"], "공유종없는_쌍_비율_d1_pct": gdm_ssdm["pct_saturated_d1"], "모형_설명력_pct": gdm_ssdm["pct_dev_explained"], "상관계수_r": gdm_ssdm["pearson_r"]},  # 체계 4 성능 요약
])  # 4개 체계 benchmark DataFrame 출력

단회 방문 위주의 원시 곤충 출현 행렬(`S >= 1`, `4,523`개 격자)은 지점 쌍의 `45.5%`에서 공유종이 전혀 없어($d_{ij} = 1$) 환경 차이와 종조성 차이의 관계를 읽어낼 수 없으므로 GDM 모형 설명력이 `0.81%`(`S >= 10` 선별 시 `2.23%`)에 그치지만, 고착성 육상식물 관측 행렬(`S >= 5`, 모형 설명력 `37.04%`, $r = 0.505$)과 139종 SSDM 적층 군집(`73.70%`, $r = 0.871$)에서는 환경 구배를 따른 연속적인 종조성 변화가 안정적으로 드러납니다. 이어서 마지막 `code cell`에서 2050년 `BAU-SSP585` 기준 전국 종조성 변화와 그 두 성분(종 교체 vs. 종 감소) 지도를 비교합니다.

In [ ]:
# [전체 비교 4-B] 2050년(BAU-SSP585) 전국 종조성 변화(GDM)와 그 두 성분(SSDM: 종 교체 vs. 종 감소) 지도 비교
comm_dir = os.path.join(DATA_DIR, "community_richness")  # 사전 산출된 전국 군집 종풍부도·종조성 변화 GeoTIFF directory 경로 지정
def read_comm_tif(fn):  # 군집 결과 GeoTIFF에서 내륙 1차원 실수형 vector를 추출하는 보조 함수 정의
    with rasterio.open(os.path.join(comm_dir, fn)) as src:  # 지정된 GeoTIFF file 열기
        return src.read(1)[mask_2d].astype(np.float64)  # 첫 번째 band에 육지 mask를 씌워 길이 84,576의 1차원 vector 반환

gdm_585 = read_comm_tif("GDM_TemporalTurnover_Poll_SSDM_BAU-SSP585.tif")  # 2020->2050 BAU-SSP585 화분매개자 GDM 2020->2050 종조성 변화 vector 읽기
sor_585 = read_comm_tif("SSDM_Baselga_BetaSor_Poll_BAU-SSP585.tif")  # 2020->2050 SSDM 전체 Sørensen 비유사도(beta_sor) vector 읽기
sim_585 = read_comm_tif("SSDM_Baselga_BetaSim_Poll_BAU-SSP585.tif")  # 2020->2050 SSDM 순수 종 교체 성분(beta_sim) vector 읽기
sne_585 = read_comm_tif("SSDM_Baselga_BetaSne_Poll_BAU-SSP585.tif")  # 2020->2050 SSDM 종 감소(중첩성) 성분(beta_sne) vector 읽기

elev_1d = bg_grid["elevation_m"].to_numpy()  # 고도대별 종조성 변화 양상 비교를 위해 전국 내륙 해발고도(m) 배열 추출
low_mask = (elev_1d < 200) & (sor_585 > 0)  # 해발 200 m 미만 저지대 유효 격자 mask 생성
high_mask = (elev_1d >= 800) & (sor_585 > 0)  # 해발 800 m 이상 고산지대 유효 격자 mask 생성
print(f"저지대 (<200 m): 평균 beta_sor = {sor_585[low_mask].mean():.3f} (종 감소 beta_sne = {sne_585[low_mask].mean():.3f}, 종 교체 beta_sim = {sim_585[low_mask].mean():.3f})")  # 저지대 평균 분할치 출력
print(f"고지대 (>=800 m): 평균 beta_sor = {sor_585[high_mask].mean():.3f} (종 감소 beta_sne = {sne_585[high_mask].mean():.3f}, 종 교체 beta_sim = {sim_585[high_mask].mean():.3f})")  # 고지대 평균 분할치 출력

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))  # GDM 종조성 변화 및 두 성분 지도를 나란히 그릴 1행 3열 figure 생성
map_specs = [  # 각 panel에 표시할 (축 객체, 자료 vector, 제목 문자열, colormap, colorbar 최댓값) 설정 list 정의
    (axes[0], gdm_585, "1. GDM 종조성 변화 (2020-2050)", "YlOrRd", 0.45),  # panel 1: GDM 총 종조성 비유사도 예측치
    (axes[1], sim_585, "2. SSDM 종 교체 (beta_sim)", "PuBuGn", 0.35),  # panel 2: Baselga Simpson 1:1 종 치환 성분
    (axes[2], sne_585, "3. SSDM 종 감소 (beta_sne)", "OrRd", 0.80),  # panel 3: Baselga 종 감소(대체 종 유입 없는 소실) 성분
]  # 3개 지도 panel 설정 list 정의 종료
for ax, arr, ttl, cm, vm in map_specs:  # 3개 지도 panel 설정을 순서대로 순회
    im = ax.imshow(vec_to_2d(arr, mask_2d), cmap=cm, vmin=0, vmax=vm)  # 1차원 지표를 2차원 남한 지도로 복원하여 지정 colormap으로 표시
    ax.set_title(f"{ttl}\n(전국 평균 {np.mean(arr):.2f})", fontsize=10.0, fontweight="bold")  # panel 상단에 지표명과 전국 평균값 표시
    ax.axis("off")  # 지도 축 테두리 눈금 비활성화
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)  # 우측에 지표 scale colorbar 추가
plt.tight_layout()  # 3개 panel 여백 정렬
plt.show()  # 전국 종조성 변화 및 두 성분 비교 지도 출력

2050년 `BAU-SSP585` 아래에서 해발 `200 m` 미만 저지대 유효 격자(`sor_585 > 0`)의 평균 전체 비유사도($\beta_{\text{sor}} = 0.753$, 전체 `<200 m` 격자 기준 `0.751`) 중 약 `85.2%`는 새로운 종의 유입 없이 기존 종이 빠져나가기만 하는 종 감소($\beta_{\text{sne}} = 0.641$, 종 교체 $\beta_{\text{sim}} = 0.112$)인 반면, 해발 `800 m` 이상 고산지대($\beta_{\text{sor}} = 0.280$)에서는 저지대·산록부 종들의 상향 유입으로 순수 종 교체($\beta_{\text{sim}} = 0.167$, `59.7%` vs. $\beta_{\text{sne}} = 0.113$)가 우세합니다.

### Key Findings (`Slides 40–41`)
- **목 수준 통합의 한계와 기능군별 차별적 취약성**: 벌목(`Hymenoptera`) 전체 통합 SDM은 `BAU-SSP585`에서 `-23.9%` 감소를 예측하지만, 기능군 단위로 분해하면 저온 적응 산지성 `FG03 뒤영벌속`(`-51.2%`, $\Delta\bar{z} = +80.9\text{ m}$, $\Delta\bar{y} = +31.3\text{ km}$)과 `FG06 단독성 야생벌류`(`-48.3%`, $\Delta\bar{z} = +106.5\text{ m}`)의 급격한 축소가 광온성 `FG04 꿀벌속`(`-23.2%`)에 가려져 있었음이 드러납니다. 감소 기능군에서 평균 해발고도 상승($\Delta\bar{z}$, `m a.s.l.`)의 `72–100%`는 고산 정상부 신규 확장이 아니라 분포 아래쪽 가장자리(저지대) 서식지 소실에 기인하며, 아열대 해안성 `FG08 큰별노린재과`는 고도 상승 없이(`-1.2 m`) 해안을 따라 수평 북상(`+24.3 km`, `+87.9%`)합니다.
- **100 m 토지피복·경관구조 변수와 부분서식지 상보성**: 5개 곤충 목의 50 km 공간 `block` 교차검증 AUC는 `0.620–0.645`(12개 기능군 `0.616–0.777`; ODMAP 기준 공간 자기상관을 제거한 전이성 지표, Zurell et al., 2020) 범위이며, 100 m 해상도에서 산출한 6개 토지피복·경관구조 변수군(`frac_0..3`, `edge_FO`, `shannon`; 특히 기능군 평균 SDM에서는 `shannon` `9.5%`·`edge_FO` `9.4%`·`frac_2` `9.2%`, SSDM-GDM에서는 `frac_0` `16.1%`·`frac_2` `9.6%`)이 화분매개자 SDM `log-odds` 가중치 합의 `38.6%`와 139종 SSDM-GDM 환경 `spline` 높이 합의 `36.4%`를 차지합니다. 이는 산림 우점 한반도에서 산림 영소처와 개방지 밀원을 잇는 임연부 경계밀도(`edge_FO`)가 핵심 서식 요인임을 보여줍니다(Westrich, 1996; Tscharntke et al., 2005).
- **여름 고온 한계($P_{99}$) 반영과 국내 자료만 쓸 때의 기후 범위 과소추정 보완**: 전 지구 GBIF(`10'` WorldClim 2.1 공간 중복 제거 $P_{99}^{\text{GBIF}}$)와 국내 NES(`1 km` 기상청 KMA $P_{99}^{\text{KR}}$) 결합 임계치를 적용하면 국내 분포만으로 기후 범위를 잘라 보는 오류(지위 절단, *niche truncation*)와 기후장 기준연도 편차를 함께 보정하여, `FG03 뒤영벌속`의 2050년 `BAU-SSP585` 감소율이 고온 한계 미반영 시 `-37.2%`에서 `33.41 °C` 고온 한계 반영 시 `-51.2%`로 포착됩니다.
- **식물–화분매개자 비대칭 불일치 및 피난처 보전 효과**: 초지 밀원 감소(`-23.9%`)가 겹치면서 `FG01 꽃등에과 × 초지 국화과·산형과 밀원군` 공존 면적은 `-34.2%`(단일 분류군 `-30.9%`)로 축소되며, 수명이 긴 목본 식물의 저지대 잔존 관성과 `FG03 뒤영벌속`·`FG09 호랑나비과`의 산지 후퇴가 엇갈리면서 2020년 공존 서식지의 `22.9–23.1%`(2050년 전체 적합 상호작용 `partner` 식물 격자의 `41.6–54.0%`)가 수분 공백에 노출됩니다(Schweiger et al., 2008; Schleuning et al., 2016). 기저 종풍부도 $S_{2020} \ge 5$ 유효 서식 격자(`76,575`개, 내륙의 `90.5%`) 기준 `Biodiversity-SSP126`은 전국 공존 피난처를 `29.9%` 유지하여 `BAU-SSP585`(`15.2%`, 해발 `200–500 m` 산록부 밀원·기주 결핍 `51.1%`) 대비 약 두 배의 상호작용 보전 효과를 보입니다.
- **반도 고립성에 따른 저지대 생물상 빈약화 vs. 고지대 종 교체**: 단회 방문 원시 곤충 기록의 공유종 부재(`45.5%`에서 $d_{ij} = 1$, 원시 GDM 설명력 `0.81%` vs. 식물 관측 행렬 `37.04%`)를 보완한 139종 SSDM-GDM(모형 설명력 `73.70%`) 및 Baselga 분할 결과, 고지대(`>=800 m`)에서는 유럽 산악 정상부(Steinbauer et al., 2018)와 유사한 상향 유입 종 교체(`59.7%`, $\beta_{\text{sim}} = 0.167$)가 우세하나, 해양으로 격리된 반도 저지대(`<200 m`)에서는 남방계 대체 종 유입 부족으로 종조성 변화의 `85.2%`가 종 감소($\beta_{\text{sne}} = 0.640$ vs. $\beta_{\text{sim}} = 0.111$)로 나타납니다.

### Further work
- **단일 기능군 민감도 비교 실습**: `[실습 1-A]`~`[실습 2-B]` `cell`의 대상 식별자를 `FG03`(`뒤영벌속`, `33.41 °C`)에서 고온 내성 `FG04`(`꿀벌속`, `41.53 °C`) 또는 남방계 `FG08`(`큰별노린재과`, 상한 `35.86 °C`, 동계 하한 `-13.96 °C`)로 바꾸거나 정규화 강도 `beta_reg`(`0.5` vs. `3.0`)를 조정하여 반응곡선과 2050년 서식지 변화 지도의 변화를 비교할 수 있습니다.
- **`Biodiversity-SSP126` 공존 피난처 완화 효과 직접 비교**: `Step 2-B`에서 이미 불러온 `env_bio126`(`env_2050_Biodiversity_SSP126`)을 `[실습 3]`·`[전체 비교 3]`의 `project_to_scenario(...)` 및 `analyze_bipartite_overlap(...)`에 적용하여, 기후 완화와 임연부·초지 보전이 결합될 때 8개 상호작용 쌍의 공존 유지율과 수분 공백 비율이 `BAU-SSP585` 대비 얼마나 개선되는지 직접 검정할 수 있습니다.